In [ ]:
!apt-get update

!apt-get install -y \
    build-essential \
    python3-dev \
    libcairo2-dev \
    libpango1.0-dev \
    pkg-config \
    ffmpeg \
    texlive \
    texlive-latex-extra \
    texlive-fonts-extra \
    texlive-latex-recommended \
    texlive-science \
    tipa

Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [113 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [44.3 kB]
Get:14 ht

In [ ]:
%pip install manim

In [ ]:
from manim import *
import numpy as np

In [ ]:
# ============================================================
# DATA
# ============================================================

def make_symbol_images():
    """
    7x7 binary representations of:
    +, -, /, *
    """

    plus = np.zeros((7, 7), dtype=int)
    plus[:, 3] = 1
    plus[3, :] = 1

    minus = np.zeros((7, 7), dtype=int)
    minus[3, :] = 1

    slash = np.zeros((7, 7), dtype=int)
    for r in range(7):
        slash[r, 6 - r] = 1

    star = np.zeros((7, 7), dtype=int)
    for r in range(7):
        star[r, r] = 1
        star[r, 6 - r] = 1
    star[:, 3] = 1

    return {
        "+": plus,
        "−": minus,
        "/": slash,
        "*": star,
    }


# ============================================================
# GENERAL GRID HELPERS
# ============================================================

def make_binary_grid(
    data,
    cell_size=0.34,
    show_values=True,
    empty=False,
    font_size=14,
):
    """
    Creates a grid where:
        0 -> black
        1 -> white

    Each cell is:
        VGroup(square, number)
    """

    rows, cols = data.shape
    cells = VGroup()

    for value in data.flatten():

        square = Square(
            side_length=cell_size,
            stroke_color=GRAY_B,
            stroke_width=1.2,
        )

        if empty:
            square.set_fill(BLACK, opacity=0)
        else:
            if value == 1:
                square.set_fill(WHITE, opacity=1)
            else:
                square.set_fill(BLACK, opacity=1)

        if value == 1:
            number_color = BLACK
        else:
            number_color = GRAY_A

        number = Text(
            str(int(value)),
            font_size=font_size,
            color=number_color,
        )

        if not show_values or empty:
            number.set_opacity(0)

        cells.add(
            VGroup(square, number)
        )

    cells.arrange_in_grid(
        rows=rows,
        cols=cols,
        buff=0,
    )

    return cells


def fill_binary_grid_animation(grid, data):
    """
    Returns animations that fill an initially empty grid.
    """

    animations = []

    for cell, value in zip(grid, data.flatten()):

        square = cell[0]
        number = cell[1]

        if value == 1:
            fill_color = WHITE
            text_color = BLACK
        else:
            fill_color = BLACK
            text_color = GRAY_A

        animations.append(
            AnimationGroup(
                square.animate.set_fill(
                    fill_color,
                    opacity=1,
                ),
                number.animate
                .set_color(text_color)
                .set_opacity(1),
            )
        )

    return animations


def get_patch_cells(
    grid,
    n_cols,
    top,
    left,
    size=3,
):
    cells = []

    for r in range(top, top + size):
        for c in range(left, left + size):
            cells.append(
                grid[r * n_cols + c]
            )

    return VGroup(*cells)


def cnn_correlation_valid(image, kernel):
    """
    CNN-style convolution.
    Technically this is cross-correlation because
    the kernel is not flipped.
    """

    H, W = image.shape
    KH, KW = kernel.shape

    output = np.zeros(
        (
            H - KH + 1,
            W - KW + 1,
        ),
        dtype=int,
    )

    for i in range(output.shape[0]):
        for j in range(output.shape[1]):

            patch = image[
                i:i + KH,
                j:j + KW,
            ]

            output[i, j] = np.sum(
                patch * kernel
            )

    return output


# ============================================================
# FEATURE MAP HELPERS
# ============================================================

def feature_color(value, max_value=3):
    """
    Binary kernels produce values from 0 to 3.
    Higher response -> stronger blue.
    """

    if value <= 0:
        return GRAY_E

    alpha = min(
        value / max_value,
        1,
    )

    return interpolate_color(
        GRAY_D,
        BLUE_C,
        alpha,
    )


def make_feature_map(
    data,
    cell_size=0.38,
    reveal=True,
    font_size=15,
):
    cells = VGroup()

    max_value = max(
        int(np.max(data)),
        1,
    )

    for value in data.flatten():

        color = feature_color(
            value,
            max_value,
        )

        square = Square(
            side_length=cell_size,
            stroke_color=GRAY_B,
            stroke_width=1.1,
        )

        if reveal:
            square.set_fill(
                color,
                opacity=0.8,
            )
        else:
            square.set_fill(
                BLACK,
                opacity=0,
            )

        number = Text(
            str(int(value)),
            font_size=font_size,
            color=WHITE,
        )

        if not reveal:
            number.set_opacity(0)

        cells.add(
            VGroup(square, number)
        )

    cells.arrange_in_grid(
        rows=data.shape[0],
        cols=data.shape[1],
        buff=0,
    )

    return cells


def reveal_feature_cell(
    cell,
    value,
    max_value=3,
):
    color = feature_color(
        value,
        max_value,
    )

    return AnimationGroup(
        cell[0].animate.set_fill(
            color,
            opacity=0.8,
        ),
        cell[1].animate.set_opacity(1),
    )

def make_multiplication_lines(patch, kernel, font_size=23):
    """
    Creates one multiplication-expression line per patch row.

    Example:
        0×0 + 1×1 + 0×0
    """

    lines = VGroup()

    for r in range(patch.shape[0]):

        expression = " + ".join(
            [
                rf"{int(patch[r, c])}\times{int(kernel[r, c])}"
                for c in range(patch.shape[1])
            ]
        )

        line = MathTex(
            expression,
            font_size=font_size,
        )

        lines.add(line)

    lines.arrange(
        DOWN,
        aligned_edge=LEFT,
        buff=0.08,
    )

    return lines


In [ ]:
# ============================================================
# VIDEO 1
# IMAGE REPRESENTATION
# ============================================================

class Video1ImageRepresentation(Scene):

    def construct(self):

        symbols = make_symbol_images()

        title = Text(
            "How does a computer represent an image?",
            font_size=40,
        ).to_edge(UP)

        self.play(Write(title))

        # ----------------------------------------------------
        # Original symbols
        # ----------------------------------------------------

        symbol_objects = VGroup(
            *[
                Text(
                    symbol,
                    font_size=75,
                )
                for symbol in symbols.keys()
            ]
        )

        symbol_objects.arrange(
            RIGHT,
            buff=2.0,
        )

        symbol_objects.shift(
            DOWN * 0.2
        )

        self.play(
            LaggedStart(
                *[
                    FadeIn(
                        s,
                        scale=0.8,
                    )
                    for s in symbol_objects
                ],
                lag_ratio=0.15,
            )
        )

        self.wait(1)

        # ----------------------------------------------------
        # Move symbols upward
        # ----------------------------------------------------

        target_x = [
            -4.8,
            -1.6,
            1.6,
            4.8,
        ]

        self.play(
            *[
                symbol.animate.move_to(
                    [x, 2.0, 0]
                )
                for symbol, x
                in zip(
                    symbol_objects,
                    target_x,
                )
            ]
        )

        # ----------------------------------------------------
        # Empty grids
        # ----------------------------------------------------

        grids = []

        for (
            symbol,
            data,
            x,
        ) in zip(
            symbols.keys(),
            symbols.values(),
            target_x,
        ):

            grid = make_binary_grid(
                data,
                cell_size=0.29,
                show_values=True,
                empty=True,
                font_size=11,
            )

            grid.move_to(
                [x, 0.1, 0]
            )

            grids.append(grid)

        grids_group = VGroup(*grids)

        self.play(
            LaggedStart(
                *[
                    FadeIn(grid)
                    for grid in grids
                ],
                lag_ratio=0.12,
            )
        )

        # ----------------------------------------------------
        # Binary explanation
        # ----------------------------------------------------

        legend = VGroup(
            Text(
                "0 = black",
                font_size=25,
            ),
            Text(
                "1 = white",
                font_size=25,
            ),
        ).arrange(
            RIGHT,
            buff=1.0,
        )

        legend.to_edge(DOWN)

        self.play(
            FadeIn(legend)
        )

        # ----------------------------------------------------
        # Fill grids
        # ----------------------------------------------------

        for grid, data in zip(
            grids,
            symbols.values(),
        ):

            animations = (
                fill_binary_grid_animation(
                    grid,
                    data,
                )
            )

            self.play(
                LaggedStart(
                    *animations,
                    lag_ratio=0.005,
                ),
                run_time=1.0,
            )

        self.wait(2)

        # ----------------------------------------------------
        # Matrix notation
        # ----------------------------------------------------

        matrix_text = MathTex(
            r"X \in \mathbb{R}^{7\times7}",
            font_size=36,
        )

        matrix_text.next_to(
            legend,
            UP,
            buff=0.4,
        )

        self.play(
            Write(matrix_text)
        )

        self.wait(2)

        self.play(
            FadeOut(
                VGroup(
                    title,
                    symbol_objects,
                    grids_group,
                    legend,
                    matrix_text,
                )
            )
        )


In [ ]:
# ============================================================
# VIDEO 2
# CONVOLUTION + INNER PRODUCT
# ============================================================

class Video2ConvolutionInnerProduct(Scene):

    def construct(self):

        image = make_symbol_images()["+"]

        vertical_kernel = np.array([
            [0, 1, 0],
            [0, 1, 0],
            [0, 1, 0],
        ])

        feature_map = cnn_correlation_valid(
            image,
            vertical_kernel,
        )

        title = Text(
            "Convolution as an Inner Product",
            font_size=39,
        ).to_edge(UP)

        self.play(Write(title))

        # ====================================================
        # MAIN LAYOUT
        #
        # Input  *  Kernel  ->  Feature Map
        # ====================================================

        input_grid = make_binary_grid(
            image,
            cell_size=0.37,
            show_values=True,
            font_size=12,
        )

        input_grid.move_to(
            [-4.7, 0.8, 0]
        )

        input_label = Text(
            "Input",
            font_size=25,
        ).next_to(
            input_grid,
            UP,
            buff=0.22,
        )

        kernel_grid = make_binary_grid(
            vertical_kernel,
            cell_size=0.37,
            show_values=True,
            font_size=13,
        )

        kernel_grid.move_to(
            [-1.55, 0.8, 0]
        )

        kernel_label = Text(
            "Kernel",
            font_size=25,
        ).next_to(
            kernel_grid,
            UP,
            buff=0.22,
        )

        convolution_symbol = MathTex(
            r"\star",
            font_size=48,
        ).move_to(
            [-3.05, 0.8, 0]
        )

        arrow = Arrow(
            start=[-0.55, 0.8, 0],
            end=[1.1, 0.8, 0],
            buff=0.05,
        )

        output_grid = make_feature_map(
            feature_map,
            cell_size=0.39,
            reveal=False,
            font_size=14,
        )

        output_grid.move_to(
            [3.15, 0.8, 0]
        )

        output_label = Text(
            "Feature Map",
            font_size=25,
        ).next_to(
            output_grid,
            UP,
            buff=0.22,
        )

        self.play(
            FadeIn(input_grid),
            FadeIn(input_label),
            FadeIn(convolution_symbol),
            FadeIn(kernel_grid),
            FadeIn(kernel_label),
            GrowArrow(arrow),
            FadeIn(output_grid),
            FadeIn(output_label),
        )

        self.wait(0.7)

        # ====================================================
        # SCAN STARTS AT THE FIRST VALID PATCH
        # ====================================================

        patch_box = SurroundingRectangle(
            get_patch_cells(
                input_grid,
                n_cols=7,
                top=0,
                left=0,
                size=3,
            ),
            color=YELLOW,
            stroke_width=4,
            buff=0.02,
        )

        self.play(Create(patch_box))

        # ----------------------------------------------------
        # Helper used for detailed patch explanations
        # ----------------------------------------------------

        def explain_patch(top, left, slow=False):

            patch_data = image[
                top:top + 3,
                left:left + 3,
            ]

            value = int(
                np.sum(
                    patch_data
                    * vertical_kernel
                )
            )

            patch_copy = make_binary_grid(
                patch_data,
                cell_size=0.28,
                show_values=True,
                font_size=11,
            )

            kernel_copy = make_binary_grid(
                vertical_kernel,
                cell_size=0.28,
                show_values=True,
                font_size=11,
            )

            patch_label = Text(
                "Patch",
                font_size=19,
            ).next_to(
                patch_copy,
                DOWN,
                buff=0.12,
            )

            kernel_copy_label = Text(
                "Kernel",
                font_size=19,
            ).next_to(
                kernel_copy,
                DOWN,
                buff=0.12,
            )

            mult = MathTex(
                r"\odot",
                font_size=34,
            )

            equals = MathTex(
                "=",
                font_size=34,
            )

            left_side = VGroup(
                patch_copy,
                mult,
                kernel_copy,
                equals,
            ).arrange(
                RIGHT,
                buff=0.22,
            )

            multiplication_lines = make_multiplication_lines(
                patch_data,
                vertical_kernel,
                font_size=22,
            )

            result = MathTex(
                rf"= {value}",
                font_size=34,
                color=YELLOW if value > 0 else WHITE,
            )

            calculation = VGroup(
                left_side,
                multiplication_lines,
                result,
            ).arrange(
                RIGHT,
                buff=0.35,
            )

            calculation.to_edge(
                DOWN,
                buff=0.35,
            )

            patch_label.next_to(
                patch_copy,
                DOWN,
                buff=0.1,
            )

            kernel_copy_label.next_to(
                kernel_copy,
                DOWN,
                buff=0.1,
            )

            patch_cells = get_patch_cells(
                input_grid,
                n_cols=7,
                top=top,
                left=left,
                size=3,
            )

            self.play(
                TransformFromCopy(
                    patch_cells,
                    patch_copy,
                ),
                TransformFromCopy(
                    kernel_grid,
                    kernel_copy,
                ),
                FadeIn(patch_label),
                FadeIn(kernel_copy_label),
                Write(mult),
                Write(equals),
                run_time=0.65,
            )

            if slow:

                for line in multiplication_lines:
                    self.play(
                        Write(line),
                        run_time=0.6,
                    )

                self.play(
                    Write(result),
                    run_time=0.6,
                )

            else:

                self.play(
                    Write(multiplication_lines),
                    Write(result),
                    run_time=0.8,
                )

            self.wait(
                0.7 if slow else 0.25
            )

            # Send result to corresponding feature-map cell

            index = (
                top * feature_map.shape[1]
                + left
            )

            target_cell = output_grid[index]

            flying_result = result.copy()

            self.play(
                flying_result.animate
                .scale(0.45)
                .move_to(
                    target_cell.get_center()
                ),
                run_time=0.65,
            )

            self.remove(flying_result)

            self.play(
                reveal_feature_cell(
                    target_cell,
                    value,
                    max_value=3,
                ),
                run_time=0.35,
            )

            self.play(
                FadeOut(
                    VGroup(
                        patch_copy,
                        kernel_copy,
                        patch_label,
                        kernel_copy_label,
                        mult,
                        equals,
                        multiplication_lines,
                        result,
                    )
                ),
                run_time=0.4,
            )

        # ====================================================
        # FIRST PATCH
        #
        # Everything is zero.
        # ====================================================

        explain_patch(
            0,
            0,
            slow=False,
        )

        # ====================================================
        # SECOND PATCH
        # Still zero, so faster.
        # ====================================================

        second_patch = get_patch_cells(
            input_grid,
            7,
            0,
            1,
            3,
        )

        self.play(
            patch_box.animate.move_to(
                second_patch.get_center()
            ),
            run_time=0.45,
        )

        second_value = int(
            feature_map[0, 1]
        )

        second_cell = output_grid[1]

        self.play(
            reveal_feature_cell(
                second_cell,
                second_value,
                max_value=3,
            ),
            run_time=0.35,
        )

        # ====================================================
        # THIRD PATCH
        #
        # This is the first non-zero response.
        # Slow down here.
        # ====================================================

        third_patch = get_patch_cells(
            input_grid,
            7,
            0,
            2,
            3,
        )

        self.play(
            patch_box.animate.move_to(
                third_patch.get_center()
            ),
            run_time=0.5,
        )

        explain_patch(
            0,
            2,
            slow=True,
        )

        self.wait(0.5)

        # ====================================================
        # CONTINUE FULL CONVOLUTION
        # ====================================================

        already_done = {
            (0, 0),
            (0, 1),
            (0, 2),
        }

        for i in range(
            feature_map.shape[0]
        ):
            for j in range(
                feature_map.shape[1]
            ):

                if (i, j) in already_done:
                    continue

                patch = get_patch_cells(
                    input_grid,
                    7,
                    i,
                    j,
                    3,
                )

                index = (
                    i * feature_map.shape[1]
                    + j
                )

                cell = output_grid[
                    index
                ]

                value = int(
                    feature_map[i, j]
                )

                self.play(
                    patch_box.animate.move_to(
                        patch.get_center()
                    ),
                    run_time=0.075,
                )

                self.play(
                    reveal_feature_cell(
                        cell,
                        value,
                        max_value=3,
                    ),
                    run_time=0.045,
                )

        self.play(
            FadeOut(patch_box)
        )

        self.wait(1.5)

        # ====================================================
        # DIFFERENT FILTERS — SIMULTANEOUS COMPARISON
        # ====================================================

        self.play(
            FadeOut(
                VGroup(
                    convolution_symbol,
                    kernel_grid,
                    kernel_label,
                    arrow,
                    output_grid,
                    output_label,
                    input_label,
                )
            )
        )

        # ----------------------------------------------------
        # Title for the comparison section
        # ----------------------------------------------------

        comparison_title = Text(
            "One image, different kernels, different feature maps",
            font_size=24,
        )

        comparison_title.next_to(
            title,
            DOWN,
            buff=0.28,
        )

        self.play(FadeIn(comparison_title))

        # ----------------------------------------------------
        # Move the input to the left and slightly downward
        # ----------------------------------------------------

        self.play(
            input_grid.animate
            .scale(0.72)
            .move_to([-5.1, -0.35, 0]),
            run_time=0.8,
        )

        same_input_label = Text(
            "Input",
            font_size=22,
        ).next_to(
            input_grid,
            UP,
            buff=0.18,
        )

        self.play(FadeIn(same_input_label))

        # ----------------------------------------------------
        # Split point / hub
        # ----------------------------------------------------

        split_point = Dot(
            point=[-2.95, -0.35, 0],
            radius=0.04,
            color=WHITE,
        )

        input_to_split = Arrow(
            start=input_grid.get_right() + RIGHT * 0.05,
            end=split_point.get_left() + LEFT * 0.04,
            buff=0.04,
            stroke_width=2.5,
            max_tip_length_to_length_ratio=0.14,
        )

        self.play(
            FadeIn(split_point),
            GrowArrow(input_to_split),
            run_time=0.7,
        )

        # ----------------------------------------------------
        # Filters and feature maps
        # ----------------------------------------------------

        filters = [
            (
                "Vertical",
                np.array([
                    [0, 1, 0],
                    [0, 1, 0],
                    [0, 1, 0],
                ]),
            ),
            (
                "Horizontal",
                np.array([
                    [0, 0, 0],
                    [1, 1, 1],
                    [0, 0, 0],
                ]),
            ),
            (
                "Diagonal /",
                np.array([
                    [0, 0, 1],
                    [0, 1, 0],
                    [1, 0, 0],
                ]),
            ),
        ]

        # More vertical spacing and slightly smaller panels
        branch_y_positions = [1.25, -0.35, -1.95]

        branches = []
        branch_arrows = []

        for (name, kernel), y_position in zip(
            filters,
            branch_y_positions,
        ):

            fmap = cnn_correlation_valid(image, kernel)

            kernel_visual = make_binary_grid(
                kernel,
                cell_size=0.22,
                show_values=True,
                font_size=9,
            )

            fmap_visual = make_feature_map(
                fmap,
                cell_size=0.22,
                reveal=True,
                font_size=8,
            )

            internal_arrow = Arrow(
                LEFT * 0.24,
                RIGHT * 0.24,
                buff=0,
                stroke_width=2.6,
                max_tip_length_to_length_ratio=0.18,
            )

            operation = VGroup(
                kernel_visual,
                internal_arrow,
                fmap_visual,
            ).arrange(
                RIGHT,
                buff=0.20,
            )

            branch_label = Text(
                name,
                font_size=19,
            )

            branch_content = VGroup(
                branch_label,
                operation,
            ).arrange(
                DOWN,
                buff=0.12,
            )

            branch_content.move_to([2.45, y_position, 0])

            panel = SurroundingRectangle(
                branch_content,
                buff=0.14,
                stroke_color=GRAY_C,
                stroke_width=1.1,
            )

            branch_group = VGroup(panel, branch_content)
            branches.append(branch_group)

            # Arrow from split point to each panel
            branch_arrow = Arrow(
                start=split_point.get_center(),
                end=panel.get_left() + LEFT * 0.06,
                buff=0.08,
                stroke_width=2.2,
                max_tip_length_to_length_ratio=0.11,
            )

            branch_arrows.append(branch_arrow)

        # ----------------------------------------------------
        # Animate arrows from split point
        # ----------------------------------------------------

        self.play(
            LaggedStart(
                *[GrowArrow(a) for a in branch_arrows],
                lag_ratio=0.15,
            ),
            run_time=1.1,
        )

        # ----------------------------------------------------
        # Reveal all three branches
        # ----------------------------------------------------

        self.play(
            LaggedStart(
                *[
                    FadeIn(branch, shift=RIGHT * 0.12)
                    for branch in branches
                ],
                lag_ratio=0.14,
            ),
            run_time=1.2,
        )

        # Time to compare the maps
        self.wait(5)

        # ====================================================
        # FINAL QUESTION — ALONE
        # ====================================================

        comparison_objects = VGroup(
            title,
            comparison_title,
            input_grid,
            same_input_label,
            split_point,
            input_to_split,
            *branch_arrows,
            *branches,
        )

        self.play(
            FadeOut(comparison_objects),
            run_time=0.9,
        )

        question = Text(
            "What is happening mathematically\nbehind the convolution?",
            font_size=39,
            line_spacing=1.15,
        )

        self.play(
            Write(question),
            run_time=1.2,
        )

        self.wait(3)

        self.play(
            FadeOut(question),
            run_time=0.7,
        )

In [ ]:
# ============================================================
# VIDEO 3
# MATRIX REPRESENTATION
# ============================================================

class Video3MatrixRepresentation(Scene):

    def construct(self):

        title = Text(
            "Convolution as Matrix Multiplication",
            font_size=39,
        ).to_edge(UP)

        self.play(
            Write(title)
        )

        # ====================================================
        # 1. SIMPLE 1D CONVOLUTION
        # ====================================================

        input_values = [
            1,
            2,
            3,
            4,
        ]

        kernel_values = [
            2,
            1,
        ]

        output_values = [
            4,
            7,
            10,
        ]

        operation_label = MathTex(
            r"x \star k",
            font_size=35,
        ).next_to(
            title,
            DOWN,
            buff=0.35,
        )

        self.play(
            Write(operation_label)
        )

        # ----------------------------------------------------
        # Input cells
        # ----------------------------------------------------

        input_cells = VGroup()

        for value in input_values:

            square = Square(
                side_length=0.75,
                stroke_color=GRAY_B,
            )

            value_text = MathTex(
                str(value),
                font_size=32,
            )

            input_cells.add(
                VGroup(
                    square,
                    value_text,
                )
            )

        input_cells.arrange(
            RIGHT,
            buff=0,
        )

        input_cells.move_to(
            [-2.0, 1.1, 0]
        )

        input_label = MathTex(
            "x=",
            font_size=34,
        ).next_to(
            input_cells,
            LEFT,
            buff=0.3,
        )

        # ----------------------------------------------------
        # Kernel cells
        # ----------------------------------------------------

        kernel_cells = VGroup()

        for value in kernel_values:

            square = Square(
                side_length=0.75,
                stroke_color=BLUE_B,
            )

            value_text = MathTex(
                str(value),
                font_size=32,
                color=BLUE_B,
            )

            kernel_cells.add(
                VGroup(
                    square,
                    value_text,
                )
            )

        kernel_cells.arrange(
            RIGHT,
            buff=0,
        )

        kernel_cells.move_to(
            [-2.75, -0.05, 0]
        )

        kernel_label = MathTex(
            "k=",
            font_size=34,
        ).next_to(
            kernel_cells,
            LEFT,
            buff=0.3,
        )

        # ----------------------------------------------------
        # Output cells
        # ----------------------------------------------------

        output_cells = VGroup()

        for _ in output_values:

            square = Square(
                side_length=0.75,
                stroke_color=GRAY_B,
            )

            output_cells.add(
                VGroup(square)
            )

        output_cells.arrange(
            RIGHT,
            buff=0,
        )

        output_cells.move_to(
            [-2.4, -1.65, 0]
        )

        output_label = MathTex(
            "y=",
            font_size=34,
        ).next_to(
            output_cells,
            LEFT,
            buff=0.3,
        )

        self.play(
            FadeIn(input_cells),
            FadeIn(input_label),
            FadeIn(kernel_cells),
            FadeIn(kernel_label),
            FadeIn(output_cells),
            FadeIn(output_label),
        )

        # ====================================================
        # FIRST CONVOLUTION — SLOW
        # ====================================================

        window = SurroundingRectangle(
            VGroup(
                input_cells[0],
                input_cells[1],
            ),
            color=YELLOW,
            buff=0.04,
            stroke_width=4,
        )

        self.play(
            Create(window)
        )

        first_equation = MathTex(
            r"1\times2"
            r"+"
            r"2\times1"
            r"="
            r"4",
            font_size=36,
        ).move_to(
            [3.1, 0.45, 0]
        )

        # Highlight the first pair

        self.play(
            Indicate(
                input_cells[0][1],
                color=YELLOW,
            ),
            Indicate(
                kernel_cells[0][1],
                color=BLUE,
            ),
        )

        self.play(
            Write(
                first_equation[0][0:3]
            ),
            run_time=0.8,
        )

        # Second pair

        self.play(
            Indicate(
                input_cells[1][1],
                color=YELLOW,
            ),
            Indicate(
                kernel_cells[1][1],
                color=BLUE,
            ),
        )

        self.play(
            Write(first_equation),
            run_time=1.0,
        )

        result_1 = MathTex(
            "4",
            font_size=32,
            color=YELLOW,
        ).move_to(
            output_cells[0]
            .get_center()
        )

        self.play(
            TransformFromCopy(
                first_equation,
                result_1,
            )
        )

        output_cells[0].add(
            result_1
        )

        self.wait(0.6)

        # ====================================================
        # SECOND CONVOLUTION — NORMAL SPEED
        # ====================================================

        self.play(
            window.animate.move_to(
                VGroup(
                    input_cells[1],
                    input_cells[2],
                ).get_center()
            ),
            run_time=0.45,
        )

        second_equation = MathTex(
            r"2\times2"
            r"+"
            r"3\times1"
            r"="
            r"7",
            font_size=36,
        ).move_to(
            first_equation.get_center()
        )

        self.play(
            ReplacementTransform(
                first_equation,
                second_equation,
            ),
            run_time=0.6,
        )

        result_2 = MathTex(
            "7",
            font_size=32,
            color=YELLOW,
        ).move_to(
            output_cells[1]
            .get_center()
        )

        self.play(
            FadeIn(result_2),
            run_time=0.3,
        )

        output_cells[1].add(
            result_2
        )

        # ====================================================
        # THIRD CONVOLUTION
        # ====================================================

        self.play(
            window.animate.move_to(
                VGroup(
                    input_cells[2],
                    input_cells[3],
                ).get_center()
            ),
            run_time=0.4,
        )

        third_equation = MathTex(
            r"3\times2"
            r"+"
            r"4\times1"
            r"="
            r"10",
            font_size=36,
        ).move_to(
            second_equation.get_center()
        )

        self.play(
            ReplacementTransform(
                second_equation,
                third_equation,
            ),
            run_time=0.5,
        )

        result_3 = MathTex(
            "10",
            font_size=32,
            color=YELLOW,
        ).move_to(
            output_cells[2]
            .get_center()
        )

        self.play(
            FadeIn(result_3),
            run_time=0.3,
        )

        output_cells[2].add(
            result_3
        )

        self.wait(1)

        # ====================================================
        # 2. MATRIX REPRESENTATION
        # ====================================================

        self.play(
            FadeOut(
                VGroup(
                    operation_label,
                    input_cells,
                    input_label,
                    kernel_cells,
                    kernel_label,
                    output_cells,
                    output_label,
                    window,
                    third_equation,
                )
            )
        )

        y_matrix = Matrix(
            [
                [4],
                [7],
                [10],
            ]
        )

        equals = MathTex("=")

        toeplitz = Matrix([
            [2, 1, 0, 0],
            [0, 2, 1, 0],
            [0, 0, 2, 1],
        ])

        x_matrix = Matrix(
            [
                [1],
                [2],
                [3],
                [4],
            ]
        )

        matrix_expression = VGroup(
            y_matrix,
            equals,
            toeplitz,
            x_matrix,
        ).arrange(
            RIGHT,
            buff=0.35,
        )

        matrix_expression.scale(
            0.68
        )

        matrix_expression.shift(
            LEFT * 2.3
        )

        self.play(
            FadeIn(matrix_expression)
        )

        # ----------------------------------------------------
        # Toeplitz colors
        # ----------------------------------------------------

        toeplitz_entries = (
            toeplitz.get_entries()
        )

        two_indices = [
            0,
            5,
            10,
        ]

        one_indices = [
            1,
            6,
            11,
        ]

        self.play(
            *[
                toeplitz_entries[i]
                .animate
                .set_color(YELLOW)
                for i in two_indices
            ],
            *[
                toeplitz_entries[i]
                .animate
                .set_color(BLUE)
                for i in one_indices
            ],
        )

        self.wait(0.7)

        # ====================================================
        # 3. SHOW EACH MATRIX ROW BECOMING A CONVOLUTION
        # ====================================================

        matrix_rows = (
            toeplitz.get_rows()
        )

        x_entries = (
            x_matrix.get_entries()
        )

        y_entries = (
            y_matrix.get_entries()
        )

        results = [
            4,
            7,
            10,
        ]

        def animate_dot_product(
            row_index,
            slow=False,
        ):

            row = matrix_rows[
                row_index
            ]

            row_box = SurroundingRectangle(
                row,
                color=YELLOW,
                buff=0.07,
                stroke_width=3,
            )

            self.play(
                Create(row_box),
                run_time=0.35,
            )

            # -----------------------------------------------
            # Build the equation using COPIES of the
            # actual matrix and input-vector entries.
            # -----------------------------------------------

            terms = []
            separators = []

            for j in range(4):

                coefficient = (
                    row[j].copy()
                    .scale(0.85)
                )

                multiplication = MathTex(
                    r"\times",
                    font_size=25,
                )

                input_value = (
                    x_entries[j]
                    .copy()
                    .scale(0.85)
                )

                term = VGroup(
                    coefficient,
                    multiplication,
                    input_value,
                ).arrange(
                    RIGHT,
                    buff=0.05,
                )

                terms.append(term)

                if j < 3:

                    separators.append(
                        MathTex(
                            "+",
                            font_size=26,
                        )
                    )

            equals_sign = MathTex(
                "=",
                font_size=27,
            )

            result = MathTex(
                str(
                    results[
                        row_index
                    ]
                ),
                font_size=29,
                color=YELLOW,
            )

            equation_parts = []

            for j in range(4):

                equation_parts.append(
                    terms[j]
                )

                if j < 3:
                    equation_parts.append(
                        separators[j]
                    )

            equation_parts.extend(
                [
                    equals_sign,
                    result,
                ]
            )

            equation_group = VGroup(
                *equation_parts
            ).arrange(
                RIGHT,
                buff=0.09,
            )

            equation_group.scale(
                0.88
            )

            equation_group.move_to(
                [3.8, 0.25, 0]
            )

            speed = (
                0.32
                if slow
                else 0.13
            )

            # -----------------------------------------------
            # Send each coefficient and input value
            # to its multiplication term.
            # -----------------------------------------------

            for j in range(4):

                self.play(
                    TransformFromCopy(
                        row[j],
                        terms[j][0],
                    ),
                    Write(
                        terms[j][1]
                    ),
                    TransformFromCopy(
                        x_entries[j],
                        terms[j][2],
                    ),
                    run_time=speed,
                )

                if j < 3:

                    self.play(
                        Write(
                            separators[j]
                        ),
                        run_time=0.10,
                    )

            self.play(
                Write(equals_sign),
                Write(result),
                run_time=0.3,
            )

            self.play(
                Indicate(
                    y_entries[
                        row_index
                    ],
                    color=YELLOW,
                ),
                run_time=0.45,
            )

            self.wait(
                0.6 if slow else 0.25
            )

            self.play(
                FadeOut(
                    equation_group
                ),
                FadeOut(
                    row_box
                ),
                run_time=0.35,
            )

        # First row slowly

        animate_dot_product(
            0,
            slow=True,
        )

        # Remaining rows faster

        animate_dot_product(
            1,
            slow=False,
        )

        animate_dot_product(
            2,
            slow=False,
        )

        self.wait(0.7)

        self.play(
            FadeOut(
                matrix_expression
            )
        )

        # ====================================================
        # 4. WIDER CONVOLUTION MATRIX
        # ====================================================

        rows = 7
        cols = 12

        matrix_data = []

        for i in range(rows):

            row = []

            for j in range(cols):

                if j == i:
                    row.append("a")

                elif j == i + 1:
                    row.append("b")

                elif j == i + 2:
                    row.append("c")

                else:
                    row.append("0")

            matrix_data.append(
                row
            )

        large_matrix = Matrix(
            matrix_data,
            element_to_mobject=
            lambda x: MathTex(
                str(x),
                font_size=21,
            ),
            h_buff=0.48,
            v_buff=0.42,
        )

        large_matrix.scale(
            0.72
        )

        large_matrix.shift(
            UP * 0.15
        )

        matrix_label = Text(
            "A larger convolution matrix",
            font_size=27,
        ).next_to(
            title,
            DOWN,
            buff=0.4,
        )

        self.play(
            FadeIn(matrix_label),
            FadeIn(large_matrix),
        )

        entries = (
            large_matrix.get_entries()
        )

        # ====================================================
        # 5. WEIGHT SHARING FIRST
        # ====================================================

        a_entries = []
        b_entries = []
        c_entries = []
        zero_entries = []
        nonzero_entries = []

        for r in range(rows):
            for c in range(cols):

                entry = entries[
                    r * cols + c
                ]

                value = (
                    matrix_data[r][c]
                )

                if value == "a":
                    a_entries.append(
                        entry
                    )
                    nonzero_entries.append(
                        entry
                    )

                elif value == "b":
                    b_entries.append(
                        entry
                    )
                    nonzero_entries.append(
                        entry
                    )

                elif value == "c":
                    c_entries.append(
                        entry
                    )
                    nonzero_entries.append(
                        entry
                    )

                else:
                    zero_entries.append(
                        entry
                    )

        self.play(
            *[
                e.animate.set_color(
                    YELLOW
                )
                for e in a_entries
            ],
            *[
                e.animate.set_color(
                    BLUE
                )
                for e in b_entries
            ],
            *[
                e.animate.set_color(
                    GREEN
                )
                for e in c_entries
            ],
            run_time=1.2,
        )

        sharing_text = Text(
            "The same weights are reused at every position.",
            font_size=27,
        ).to_edge(
            DOWN
        )

        self.play(
            FadeIn(sharing_text)
        )

        self.wait(1.8)

        # ====================================================
        # 6. SPARSITY LAST
        #
        # This becomes the hook for the next topic.
        # ====================================================

        self.play(
            FadeOut(sharing_text)
        )

        # Return non-zero values to white first

        self.play(
            *[
                e.animate.set_color(
                    WHITE
                )
                for e in nonzero_entries
            ],
            run_time=0.6,
        )

        zero_count = len(
            zero_entries
        )

        nonzero_count = len(
            nonzero_entries
        )

        zero_tracker = ValueTracker(
            0
        )

        nonzero_tracker = ValueTracker(
            0
        )

        zero_number = Integer(
            0
        ).scale(0.75)

        nonzero_number = Integer(
            0
        ).scale(0.75)

        zero_number.add_updater(
            lambda m:
            m.set_value(
                int(
                    zero_tracker
                    .get_value()
                )
            )
        )

        nonzero_number.add_updater(
            lambda m:
            m.set_value(
                int(
                    nonzero_tracker
                    .get_value()
                )
            )
        )

        zero_label = Text(
            "Zero entries:",
            font_size=24,
        )

        nonzero_label = Text(
            "Non-zero entries:",
            font_size=24,
        )

        zero_counter = VGroup(
            zero_label,
            zero_number,
        ).arrange(
            RIGHT,
            buff=0.2,
        )

        nonzero_counter = VGroup(
            nonzero_label,
            nonzero_number,
        ).arrange(
            RIGHT,
            buff=0.2,
        )

        counters = VGroup(
            zero_counter,
            nonzero_counter,
        ).arrange(
            RIGHT,
            buff=1.0,
        )

        counters.to_edge(
            DOWN
        )

        self.play(
            FadeIn(counters)
        )

        # Highlight zeros while counting

        self.play(
            LaggedStart(
                *[
                    entry.animate
                    .set_color(YELLOW)
                    for entry
                    in zero_entries
                ],
                lag_ratio=0.012,
            ),
            *[
                entry.animate
                .set_opacity(0.25)
                for entry
                in nonzero_entries
            ],
            zero_tracker.animate
            .set_value(
                zero_count
            ),
            nonzero_tracker.animate
            .set_value(
                nonzero_count
            ),
            run_time=2.6,
        )

        sparsity_text = Text(
            "Most entries are zero.",
            font_size=29,
        ).next_to(
            counters,
            UP,
            buff=0.25,
        )

        self.play(
            FadeIn(
                sparsity_text
            )
        )

        self.wait(3)

        # Stop Integer updaters before leaving

        zero_number.clear_updaters()
        nonzero_number.clear_updaters()

        self.play(
            FadeOut(
                VGroup(
                    title,
                    matrix_label,
                    large_matrix,
                    counters,
                    sparsity_text,
                )
            )
        )

In [ ]:
# ============================================================
# ADDITIONAL HELPERS
# Used by Video 4 and Video 5.
#
# Running example, shared by both:
#
#   X = [[1, 2, 3],     K = [[1, 2],     Y = [[37, 47],
#        [4, 5, 6],          [3, 4]]          [67, 77]]
#        [7, 8, 9]]
# ============================================================


RUNNING_INPUT = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])

RUNNING_KERNEL = np.array([
    [1, 2],
    [3, 4],
])

OUTPUT_GRADIENT = np.array([
    [1, 2],
    [3, 1],
])


def make_value_grid(
    data,
    cell_size=0.6,
    font_size=24,
    text_color=WHITE,
    stroke_color=GRAY_B,
    show_values=True,
):
    """
    Numeric counterpart of make_binary_grid.

    Each cell is:
        VGroup(square, number)

    so get_patch_cells works on it unchanged.
    """

    rows, cols = data.shape
    cells = VGroup()

    for value in data.flatten():

        square = Square(
            side_length=cell_size,
            stroke_color=stroke_color,
            stroke_width=1.2,
        )

        square.set_fill(BLACK, opacity=0)

        number = MathTex(
            str(int(value)),
            font_size=font_size,
            color=text_color,
        )

        if not show_values:
            number.set_opacity(0)

        cells.add(
            VGroup(square, number)
        )

    cells.arrange_in_grid(
        rows=rows,
        cols=cols,
        buff=0,
    )

    return cells


def get_grid_row(grid, row_index, n_cols):
    """
    One full row of a grid built by make_value_grid.
    """

    return VGroup(
        *[
            grid[row_index * n_cols + c]
            for c in range(n_cols)
        ]
    )


def set_grid_values_animation(
    grid,
    data,
    font_size=24,
    color=WHITE,
):
    """
    Returns animations that rewrite every number in an existing grid.
    Used by the col2im accumulation in Video 5.
    """

    animations = []

    for cell, value in zip(grid, data.flatten()):

        new_number = MathTex(
            str(int(value)),
            font_size=font_size,
            color=color,
        )

        new_number.move_to(
            cell[0].get_center()
        )

        animations.append(
            Transform(cell[1], new_number)
        )

    return animations


def build_im2col(image, kernel_shape):
    """
    Each patch becomes one ROW.

    Shape: (H_out * W_out, KH * KW)
    """

    H, W = image.shape
    KH, KW = kernel_shape

    rows = []

    for i in range(H - KH + 1):
        for j in range(W - KW + 1):
            rows.append(
                image[i:i + KH, j:j + KW].flatten()
            )

    return np.array(rows, dtype=int)


def build_convolution_matrix(image_shape, kernel):
    """
    The sparse operator A such that

        vec(Y) = A @ vec(X)

    This is the 2D version of the Toeplitz matrix that was
    built by hand in Video 3.

    Shape: (H_out * W_out, H * W)
    """

    H, W = image_shape
    KH, KW = kernel.shape

    H_out = H - KH + 1
    W_out = W - KW + 1

    A = np.zeros(
        (H_out * W_out, H * W),
        dtype=int,
    )

    for oi in range(H_out):
        for oj in range(W_out):

            output_index = oi * W_out + oj

            for ki in range(KH):
                for kj in range(KW):

                    input_index = (
                        (oi + ki) * W
                        + (oj + kj)
                    )

                    A[output_index, input_index] = (
                        kernel[ki, kj]
                    )

    return A


def make_labeled_matrix(
    data,
    font_size=22,
    h_buff=0.62,
    v_buff=0.58,
):
    """
    Thin wrapper so every Matrix in these scenes looks the same.
    """

    return Matrix(
        [
            [str(int(v)) for v in row]
            for row in data
        ],
        element_to_mobject=lambda s: MathTex(
            s,
            font_size=font_size,
        ),
        h_buff=h_buff,
        v_buff=v_buff,
    )


# ------------------------------------------------------------
# Sanity checks. If any of these fail, the scenes below would
# be animating the wrong numbers.
# ------------------------------------------------------------

_Y = cnn_correlation_valid(RUNNING_INPUT, RUNNING_KERNEL)
_X_col = build_im2col(RUNNING_INPUT, RUNNING_KERNEL.shape)
_A = build_convolution_matrix(RUNNING_INPUT.shape, RUNNING_KERNEL)
_w = RUNNING_KERNEL.flatten()

assert np.array_equal((_X_col @ _w).reshape(2, 2), _Y)
assert np.array_equal((_A @ RUNNING_INPUT.flatten()).reshape(2, 2), _Y)


def dim_matrix_zeros(matrix_mobject, data, color=GRAY_D):
    """
    Greys out the zero entries of a Matrix so the non-zero band
    of a convolution operator stands out. Returns the matrix.
    """

    entries = matrix_mobject.get_entries()
    rows, cols = data.shape

    for r in range(rows):
        for c in range(cols):
            if int(data[r, c]) == 0:
                entries[r * cols + c].set_color(color)

    return matrix_mobject


In [ ]:
# ============================================================
# VIDEO 4
# im2col AND GEMM
# ============================================================

class Video4Im2colGemm(Scene):

    def construct(self):

        image = RUNNING_INPUT
        kernel = RUNNING_KERNEL

        feature_map = cnn_correlation_valid(image, kernel)
        image_columns = build_im2col(image, kernel.shape)

        n_patches, n_columns = image_columns.shape

        title = Text(
            "How is a convolution actually computed?",
            font_size=36,
        ).to_edge(UP, buff=0.45)

        self.play(Write(title))

        # ====================================================
        # 1. THE THREE OBJECTS, CENTRE STAGE
        # ====================================================

        input_grid = make_value_grid(
            image,
            cell_size=0.7,
            font_size=28,
        )

        kernel_grid = make_value_grid(
            kernel,
            cell_size=0.7,
            font_size=28,
            text_color=BLUE_B,
            stroke_color=BLUE_B,
        )

        output_grid = make_value_grid(
            feature_map,
            cell_size=0.82,
            font_size=28,
            text_color=YELLOW,
            stroke_color=YELLOW,
        )

        star = MathTex(r"\star", font_size=44)
        arrow = MathTex(r"\rightarrow", font_size=52)

        top_row = VGroup(
            input_grid,
            star,
            kernel_grid,
            arrow,
            output_grid,
        ).arrange(RIGHT, buff=0.55)

        top_row.move_to([0, 0.15, 0])

        input_label = MathTex("X", font_size=32)
        kernel_label = MathTex("K", font_size=32, color=BLUE_B)
        output_label = MathTex("Y", font_size=32, color=YELLOW)

        for label, grid in [
            (input_label, input_grid),
            (kernel_label, kernel_grid),
            (output_label, output_grid),
        ]:
            label.next_to(grid, UP, buff=0.22)

        labels = VGroup(
            input_label,
            kernel_label,
            output_label,
        )

        self.play(
            LaggedStart(
                FadeIn(input_grid),
                FadeIn(input_label),
                FadeIn(star),
                FadeIn(kernel_grid),
                FadeIn(kernel_label),
                FadeIn(arrow),
                FadeIn(output_grid),
                FadeIn(output_label),
                lag_ratio=0.12,
            ),
            run_time=2.0,
        )

        self.wait(1.0)

        # ====================================================
        # 2. THE PROBLEM STATEMENT
        # ====================================================

        problem = Text(
            "Hardware is fast at exactly one thing:\n"
            "dense matrix multiplication.",
            font_size=30,
            line_spacing=1.15,
        ).move_to([0, -2.2, 0])

        self.play(FadeIn(problem, shift=UP * 0.2))
        self.wait(2.2)
        self.play(FadeOut(problem))

        # ----------------------------------------------------
        # Make room: the three objects move up and shrink.
        # ----------------------------------------------------

        header = VGroup(top_row, labels)

        self.play(
            header.animate.scale(0.72).move_to([0, 2.0, 0]),
            run_time=1.0,
        )

        # ====================================================
        # 3. BUILD X_col, ONE PATCH AT A TIME
        # ====================================================

        section_text = Text(
            "Every patch becomes one row",
            font_size=28,
        ).move_to([0, 0.75, 0])

        self.play(FadeIn(section_text))

        columns_grid = make_value_grid(
            image_columns,
            cell_size=0.55,
            font_size=24,
            show_values=False,
        )

        columns_grid.move_to([0, -1.5, 0])

        columns_label = MathTex(
            r"X_{\text{col}}",
            font_size=30,
        ).next_to(columns_grid, UP, buff=0.2)

        columns_block = VGroup(columns_grid, columns_label)

        self.play(
            FadeIn(columns_grid),
            FadeIn(columns_label),
        )

        patch_box = SurroundingRectangle(
            get_patch_cells(input_grid, 3, 0, 0, 2),
            color=YELLOW,
            stroke_width=4,
            buff=0.02,
        )

        self.play(Create(patch_box))

        for index in range(n_patches):

            top = index // 2
            left = index % 2

            patch_cells = get_patch_cells(
                input_grid, 3, top, left, 2
            )

            target_cells = get_grid_row(
                columns_grid, index, n_columns
            )

            row_box = SurroundingRectangle(
                target_cells,
                color=YELLOW,
                stroke_width=3,
                buff=0.02,
            )

            self.play(
                patch_box.animate.move_to(
                    patch_cells.get_center()
                ),
                Create(row_box),
                run_time=0.5 if index == 0 else 0.35,
            )

            # Copies of the four values fly into the row,
            # in row-major order.

            flying = VGroup()
            flights = []

            for k, cell in enumerate(patch_cells):

                copy = cell[1].copy()
                flying.add(copy)

                flights.append(
                    copy.animate
                    .scale(0.85)
                    .move_to(target_cells[k][0].get_center())
                )

            self.add(flying)

            self.play(
                LaggedStart(*flights, lag_ratio=0.12),
                run_time=0.9 if index == 0 else 0.55,
            )

            # Hand over from the flying copies to the real cells.

            self.play(
                *[
                    target_cells[k][1].animate.set_opacity(1)
                    for k in range(n_columns)
                ],
                FadeOut(flying, run_time=0.01),
                run_time=0.3,
            )

            self.remove(flying)

            self.play(FadeOut(row_box), run_time=0.2)

        self.play(FadeOut(patch_box))
        self.wait(0.6)

        # ====================================================
        # 4. THE KERNEL BECOMES A COLUMN VECTOR
        # ====================================================

        self.play(
            FadeOut(section_text),
            columns_block.animate.move_to([-1.45, -1.35, 0]),
            run_time=0.8,
        )

        times = MathTex(r"\cdot", font_size=44)

        weight_vector = make_value_grid(
            kernel.reshape(4, 1),
            cell_size=0.55,
            font_size=24,
            text_color=BLUE_B,
            stroke_color=BLUE_B,
        )

        equals = MathTex("=", font_size=40)

        result_vector = make_value_grid(
            feature_map.reshape(4, 1),
            cell_size=0.55,
            font_size=24,
            text_color=YELLOW,
            stroke_color=YELLOW,
            show_values=False,
        )

        product_row = VGroup(
            times,
            weight_vector,
            equals,
            result_vector,
        ).arrange(RIGHT, buff=0.3)

        product_row.next_to(columns_grid, RIGHT, buff=0.35)

        weight_label = MathTex(
            "w", font_size=30, color=BLUE_B
        ).next_to(weight_vector, UP, buff=0.2)

        result_label = MathTex(
            r"\mathrm{vec}(Y)", font_size=28, color=YELLOW
        ).next_to(result_vector, UP, buff=0.2)

        self.play(
            Write(times),
            TransformFromCopy(kernel_grid, weight_vector),
            FadeIn(weight_label),
            run_time=1.0,
        )

        self.play(
            Write(equals),
            FadeIn(result_vector),
            FadeIn(result_label),
            run_time=0.6,
        )

        self.wait(0.5)

        # ====================================================
        # 5. ONE ROW AT A TIME, SPELLED OUT ONCE
        # ====================================================

        equation_anchor = [0, -3.15, 0]

        for index in range(n_patches):

            target_cells = get_grid_row(
                columns_grid, index, n_columns
            )

            row_box = SurroundingRectangle(
                target_cells,
                color=YELLOW,
                stroke_width=3,
                buff=0.02,
            )

            weight_box = SurroundingRectangle(
                weight_vector,
                color=BLUE_B,
                stroke_width=3,
                buff=0.02,
            )

            expression = " + ".join(
                rf"{int(image_columns[index, c])}"
                rf"\times {int(kernel.flatten()[c])}"
                for c in range(n_columns)
            )

            equation = MathTex(
                expression
                + rf" = {int(feature_map.flatten()[index])}",
                font_size=26,
            ).move_to(equation_anchor)

            self.play(
                Create(row_box),
                Create(weight_box),
                run_time=0.4,
            )

            self.play(
                Write(equation),
                run_time=1.1 if index == 0 else 0.5,
            )

            self.play(
                result_vector[index][1].animate.set_opacity(1),
                Indicate(
                    output_grid[index],
                    color=YELLOW,
                    scale_factor=1.3,
                ),
                run_time=0.6,
            )

            self.wait(0.5 if index == 0 else 0.15)

            self.play(
                FadeOut(row_box),
                FadeOut(weight_box),
                FadeOut(equation),
                run_time=0.3,
            )

        self.wait(0.8)

        # ====================================================
        # 6. THE TWO MATRICISATIONS
        #
        # Left  = the operator of Video 3, now in 2D.
        # Right = what im2col just built.
        # ====================================================

        self.play(
            FadeOut(header),
            FadeOut(columns_grid),
            FadeOut(columns_label),
            FadeOut(product_row),
            FadeOut(weight_label),
            FadeOut(result_label),
        )

        A = build_convolution_matrix(image.shape, kernel)

        left_block = VGroup(
            MathTex(
                r"\mathrm{vec}(Y) = A\,\mathrm{vec}(X)",
                font_size=30,
            ),
            dim_matrix_zeros(
                make_labeled_matrix(
                    A, font_size=24, h_buff=0.5, v_buff=0.5
                ),
                A,
            ),
            Text(
                "expand the operator,\nkeep the data",
                font_size=21,
                line_spacing=1.1,
                color=GRAY_A,
            ),
        ).arrange(DOWN, buff=0.4)

        right_block = VGroup(
            MathTex(
                r"\mathrm{vec}(Y) = X_{\text{col}}\,w",
                font_size=30,
            ),
            make_labeled_matrix(
                image_columns,
                font_size=24,
                h_buff=0.5,
                v_buff=0.5,
            ),
            Text(
                "expand the data,\nkeep the operator",
                font_size=21,
                line_spacing=1.1,
                color=GRAY_A,
            ),
        ).arrange(DOWN, buff=0.4)

        left_block.move_to([-3.5, 0.2, 0])
        right_block.move_to([3.5, 0.2, 0])

        divider = Line(
            [0, 2.15, 0],
            [0, -1.9, 0],
            stroke_color=GRAY_D,
            stroke_width=1.5,
        )

        self.play(
            FadeIn(left_block, shift=RIGHT * 0.15),
            Create(divider),
            FadeIn(right_block, shift=LEFT * 0.15),
            run_time=1.2,
        )

        self.wait(2.5)

        punchline = Text(
            "Two matricisations of the same linear operator.",
            font_size=27,
        ).move_to([0, -2.55, 0])

        self.play(FadeIn(punchline))
        self.wait(3)

        self.play(
            FadeOut(
                VGroup(
                    title,
                    left_block,
                    divider,
                    right_block,
                    punchline,
                )
            )
        )


In [ ]:
# ============================================================
# VIDEO 5
# BACKPROPAGATION AS THE TRANSPOSE
# ============================================================

class Video5BackpropTranspose(Scene):

    def construct(self):

        image = RUNNING_INPUT
        kernel = RUNNING_KERNEL
        gradient = OUTPUT_GRADIENT

        A = build_convolution_matrix(image.shape, kernel)

        input_gradient = (
            A.T @ gradient.flatten()
        ).reshape(3, 3)

        weight_gradient = cnn_correlation_valid(image, gradient)

        title = Text(
            "Backpropagation is the transpose",
            font_size=36,
        ).to_edge(UP, buff=0.45)

        self.play(Write(title))

        # ====================================================
        # 1. FORWARD AND BACKWARD
        # ====================================================

        forward = MathTex(
            r"\text{forward:}\quad",
            r"\mathrm{vec}(Y) = A\,\mathrm{vec}(X)",
            font_size=42,
        )

        backward = MathTex(
            r"\text{backward:}\quad",
            r"\frac{\partial L}{\partial\,\mathrm{vec}(X)}"
            r" = A^{\top}\,"
            r"\frac{\partial L}{\partial\,\mathrm{vec}(Y)}",
            font_size=42,
        )

        equations = VGroup(forward, backward).arrange(
            DOWN, buff=0.9, aligned_edge=LEFT
        )

        equations.move_to([0, 0.5, 0])

        self.play(Write(forward), run_time=1.2)
        self.wait(0.8)
        self.play(Write(backward), run_time=1.6)

        self.play(
            forward[1].animate.set_color(YELLOW),
            backward[1].animate.set_color(YELLOW),
            run_time=0.6,
        )

        self.play(
            forward[1].animate.set_color(WHITE),
            backward[1].animate.set_color(WHITE),
            run_time=0.6,
        )

        chain_rule = Text(
            "The Jacobian of a linear map is the map itself.",
            font_size=27,
            color=GRAY_A,
        ).move_to([0, -2.1, 0])

        self.play(FadeIn(chain_rule))
        self.wait(2.5)

        self.play(
            FadeOut(equations),
            FadeOut(chain_rule),
        )

        # ====================================================
        # 2. A BECOMES ITS TRANSPOSE
        # ====================================================

        matrix_A = dim_matrix_zeros(
            make_labeled_matrix(
                A, font_size=24, h_buff=0.6, v_buff=0.6
            ),
            A,
        )

        matrix_A.move_to([-3.55, -0.15, 0])

        label_A = MathTex("A", font_size=32).next_to(
            matrix_A, UP, buff=0.25
        )

        shape_A = MathTex(
            r"4\times 9", font_size=24, color=GRAY_B
        ).next_to(matrix_A, DOWN, buff=0.25)

        self.play(
            FadeIn(matrix_A),
            FadeIn(label_A),
            FadeIn(shape_A),
        )

        self.wait(0.8)

        matrix_AT = dim_matrix_zeros(
            make_labeled_matrix(
                A.T, font_size=24, h_buff=0.6, v_buff=0.45
            ),
            A.T,
        )

        matrix_AT.move_to([4.3, -0.15, 0])

        label_AT = MathTex(
            r"A^{\top}", font_size=32
        ).next_to(matrix_AT, UP, buff=0.25)

        shape_AT = MathTex(
            r"9\times 4", font_size=24, color=GRAY_B
        ).next_to(matrix_AT, DOWN, buff=0.25)

        transpose_arrow = MathTex(
            r"\longrightarrow", font_size=40
        ).move_to([1.35, -0.15, 0])

        transpose_caption = MathTex(
            r"(\,\cdot\,)^{\top}", font_size=26, color=GRAY_B
        ).next_to(transpose_arrow, UP, buff=0.15)

        self.play(
            FadeIn(transpose_arrow),
            FadeIn(transpose_caption),
            FadeIn(label_AT),
            FadeIn(shape_AT),
            FadeIn(matrix_AT.get_brackets()),
        )

        rows_A = matrix_A.get_rows()
        columns_AT = matrix_AT.get_columns()

        for index in range(len(rows_A)):

            row_box = SurroundingRectangle(
                rows_A[index],
                color=YELLOW,
                stroke_width=3,
                buff=0.06,
            )

            self.play(Create(row_box), run_time=0.3)

            self.play(
                TransformFromCopy(
                    rows_A[index],
                    columns_AT[index],
                ),
                run_time=0.8,
            )

            self.play(FadeOut(row_box), run_time=0.2)

        self.wait(1.5)

        self.play(
            FadeOut(
                VGroup(
                    matrix_A,
                    label_A,
                    shape_A,
                    matrix_AT,
                    label_AT,
                    shape_AT,
                    transpose_arrow,
                    transpose_caption,
                )
            )
        )

        # ====================================================
        # 3. col2im: THE TRANSPOSE IS A SCATTER-ADD
        # ====================================================

        scatter_title = Text(
            "What that transpose does, in pictures",
            font_size=27,
            color=GRAY_A,
        ).next_to(title, DOWN, buff=0.3)

        self.play(FadeIn(scatter_title))

        gradient_grid = make_value_grid(
            gradient,
            cell_size=0.66,
            font_size=26,
            text_color=YELLOW,
            stroke_color=YELLOW,
        )

        kernel_grid = make_value_grid(
            kernel,
            cell_size=0.66,
            font_size=26,
            text_color=BLUE_B,
            stroke_color=BLUE_B,
        )

        scatter_arrow = MathTex(r"\longrightarrow", font_size=44)

        accumulator = np.zeros((3, 3), dtype=int)

        accumulator_grid = make_value_grid(
            accumulator,
            cell_size=0.75,
            font_size=26,
        )

        stage = VGroup(
            gradient_grid,
            kernel_grid,
            scatter_arrow,
            accumulator_grid,
        ).arrange(RIGHT, buff=0.8)

        stage.move_to([0, 0.25, 0])

        gradient_label = MathTex(
            r"\partial L / \partial Y", font_size=26, color=YELLOW
        ).next_to(gradient_grid, UP, buff=0.25)

        kernel_label = MathTex(
            "K", font_size=26, color=BLUE_B
        ).next_to(kernel_grid, UP, buff=0.25)

        accumulator_label = MathTex(
            r"\partial L / \partial X", font_size=26
        ).next_to(accumulator_grid, UP, buff=0.25)

        self.play(
            FadeIn(stage),
            FadeIn(gradient_label),
            FadeIn(kernel_label),
            FadeIn(accumulator_label),
        )

        self.wait(0.6)

        for index in range(4):

            oi = index // 2
            oj = index % 2

            scalar = int(gradient[oi, oj])

            source_box = SurroundingRectangle(
                gradient_grid[index],
                color=YELLOW,
                stroke_width=3,
                buff=0.02,
            )

            target_cells = get_patch_cells(
                accumulator_grid, 3, oi, oj, 2
            )

            target_box = SurroundingRectangle(
                target_cells,
                color=YELLOW,
                stroke_width=3,
                buff=0.02,
            )

            contribution = MathTex(
                rf"+\;{scalar}\times K",
                font_size=32,
                color=BLUE_B,
            ).move_to([0, -2.15, 0])

            self.play(
                Create(source_box),
                Create(target_box),
                FadeIn(contribution),
                run_time=0.6,
            )

            accumulator[oi:oi + 2, oj:oj + 2] += scalar * kernel

            self.play(
                *set_grid_values_animation(
                    accumulator_grid, accumulator, font_size=26
                ),
                run_time=0.7,
            )

            self.wait(0.35 if index == 0 else 0.15)

            self.play(
                FadeOut(source_box),
                FadeOut(target_box),
                FadeOut(contribution),
                run_time=0.3,
            )

        # The scatter-add must reproduce A^T g exactly.

        assert np.array_equal(accumulator, input_gradient)

        overlap_text = Text(
            "Overlapping windows add up.\n"
            "That sum is the transpose.",
            font_size=28,
            line_spacing=1.15,
        ).move_to([0, -2.4, 0])

        self.play(FadeIn(overlap_text, shift=UP * 0.15))
        self.wait(3)

        self.play(
            FadeOut(
                VGroup(
                    scatter_title,
                    stage,
                    gradient_label,
                    kernel_label,
                    accumulator_label,
                    overlap_text,
                )
            )
        )

        # ====================================================
        # 4. THE KERNEL GRADIENT
        # ====================================================

        weight_title = Text(
            "And the gradient of the kernel?",
            font_size=27,
            color=GRAY_A,
        ).next_to(title, DOWN, buff=0.3)

        self.play(FadeIn(weight_title))

        weight_equation = MathTex(
            r"\frac{\partial L}{\partial K}"
            r"\;=\;X \star \frac{\partial L}{\partial Y}",
            font_size=42,
        ).move_to([0, 1.0, 0])

        self.play(Write(weight_equation), run_time=1.2)

        weight_grid = make_value_grid(
            weight_gradient,
            cell_size=0.75,
            font_size=27,
            text_color=BLUE_B,
            stroke_color=BLUE_B,
        ).move_to([0, -0.9, 0])

        self.play(FadeIn(weight_grid))

        weight_caption = Text(
            "Another correlation. Same operator, new argument.",
            font_size=26,
            color=GRAY_A,
        ).move_to([0, -2.5, 0])

        self.play(FadeIn(weight_caption))
        self.wait(3)

        self.play(
            FadeOut(
                VGroup(
                    weight_title,
                    weight_equation,
                    weight_grid,
                    weight_caption,
                )
            )
        )

        # ====================================================
        # 5. CLOSING
        # ====================================================

        closing = VGroup(
            MathTex(
                r"\text{forward:}\quad x \;\mapsto\; A x",
                font_size=38,
            ),
            MathTex(
                r"\text{backward:}\quad g \;\mapsto\; A^{\top} g",
                font_size=38,
            ),
            Text(
                "Training is choosing A inside the set\n"
                "of structured matrices.",
                font_size=28,
                line_spacing=1.15,
            ),
        ).arrange(DOWN, buff=0.7)

        closing.move_to([0, 0, 0])

        self.play(
            FadeOut(title),
            FadeIn(closing, shift=UP * 0.2),
            run_time=1.2,
        )

        self.wait(4)

        self.play(FadeOut(closing))


In [ ]:
# ============================================================
# COMPLETE HELPERS FOR THE FINAL CNN VIDEO
# ============================================================


# ============================================================
# NUMERICAL CNN
# ============================================================

def final_conv_valid(x, k, b=0.0):
    """
    Valid CNN-style convolution / cross-correlation.
    7x7 input with 3x3 kernel -> 5x5 feature map.
    """

    x = np.asarray(x, dtype=float)
    k = np.asarray(k, dtype=float)

    h, w = x.shape
    kh, kw = k.shape

    y = np.zeros(
        (
            h - kh + 1,
            w - kw + 1,
        ),
        dtype=float,
    )

    for i in range(y.shape[0]):
        for j in range(y.shape[1]):

            patch = x[
                i:i + kh,
                j:j + kw,
            ]

            y[i, j] = (
                np.sum(
                    patch * k
                )
                + b
            )

    return y


def final_avg_pool_2x2_stride2(x):
    """
    Average Pooling 2x2, stride 2.

    With the current 5x5 feature map:
        5x5 -> 2x2

    The final row/column are ignored.
    """

    x = np.asarray(
        x,
        dtype=float,
    )

    return np.array([
        [
            np.mean(x[0:2, 0:2]),
            np.mean(x[0:2, 2:4]),
        ],
        [
            np.mean(x[2:4, 0:2]),
            np.mean(x[2:4, 2:4]),
        ],
    ])


def final_softmax(z):
    z = np.asarray(
        z,
        dtype=float,
    )

    # Numerically stable softmax
    exp = np.exp(
        z - np.max(z)
    )

    return (
        exp
        / np.sum(exp)
    )


def final_forward(
    x,
    k,
    b,
    w,
    c,
):
    """
    Complete forward pass:

        Input
          ->
        Convolution + Bias
          ->
        ReLU
          ->
        Average Pooling
          ->
        Flatten
          ->
        Dense
          ->
        Softmax
    """

    # Convolution + bias
    z = final_conv_valid(
        x,
        k,
        b,
    )

    # ReLU
    a = np.maximum(
        z,
        0.0,
    )

    # Pooling
    p = final_avg_pool_2x2_stride2(
        a
    )

    # Flatten
    h = p.flatten()

    # Dense layer
    logits = (
        w @ h
        + c
    )

    # Softmax
    probs = final_softmax(
        logits
    )

    return {
        "z": z,
        "a": a,
        "pool": p,
        "h": h,
        "logits": logits,
        "probs": probs,
    }


def final_backward(
    x,
    label,
    k,
    b,
    w,
    c,
):
    """
    Backpropagation for the simplified CNN.

    Returns:
        forward
        delta
        dK
        db
        dW
        dc
        dZ
    """

    forward = final_forward(
        x,
        k,
        b,
        w,
        c,
    )

    # ========================================================
    # SOFTMAX + CROSS ENTROPY
    #
    # dL/dlogits = p - y
    # ========================================================

    delta = forward[
        "probs"
    ].copy()

    delta[label] -= 1.0

    # ========================================================
    # DENSE LAYER
    # ========================================================

    dW = np.outer(
        delta,
        forward["h"],
    )

    dc = delta.copy()

    dh = (
        w.T
        @ delta
    )

    # ========================================================
    # FLATTEN -> POOL
    # ========================================================

    dP = dh.reshape(
        2,
        2,
    )

    # ========================================================
    # AVERAGE POOLING BACKWARD
    #
    # Each gradient is divided equally among
    # the four values of its pooling window.
    # ========================================================

    dA = np.zeros_like(
        forward["a"]
    )

    dA[0:2, 0:2] += (
        dP[0, 0]
        / 4.0
    )

    dA[0:2, 2:4] += (
        dP[0, 1]
        / 4.0
    )

    dA[2:4, 0:2] += (
        dP[1, 0]
        / 4.0
    )

    dA[2:4, 2:4] += (
        dP[1, 1]
        / 4.0
    )

    # ========================================================
    # RELU BACKWARD
    # ========================================================

    dZ = (
        dA
        * (
            forward["z"]
            > 0
        )
    )

    # ========================================================
    # CONVOLUTION BACKWARD — GRADIENT OF K
    #
    # dK += dZ[i,j] * corresponding input patch
    # ========================================================

    dK = np.zeros_like(k)

    for i in range(
        dZ.shape[0]
    ):
        for j in range(
            dZ.shape[1]
        ):

            patch = x[
                i:i + k.shape[0],
                j:j + k.shape[1],
            ]

            dK += (
                dZ[i, j]
                * patch
            )

    # Bias gradient
    db = np.sum(
        dZ
    )

    return (
        forward,
        delta,
        dK,
        db,
        dW,
        dc,
        dZ,
    )


def final_train(
    samples,
    labels,
    k,
    b,
    w,
    c,
    epochs=800,
    lr=0.08,
):
    """
    Small batch-gradient-descent loop used only
    to produce trained parameters for the final
    demonstration with +, -, / and *.
    """

    k = k.copy()
    w = w.copy()
    c = c.copy()

    b = float(b)

    for _ in range(
        epochs
    ):

        gK = np.zeros_like(k)
        gb = 0.0

        gW = np.zeros_like(w)
        gc = np.zeros_like(c)

        for x, label in zip(
            samples,
            labels,
        ):

            (
                _,
                _,
                dK,
                db,
                dW,
                dc,
                _,
            ) = final_backward(
                x,
                label,
                k,
                b,
                w,
                c,
            )

            gK += dK
            gb += db
            gW += dW
            gc += dc

        scale = (
            1.0
            / len(samples)
        )

        k -= (
            lr
            * scale
            * gK
        )

        b -= (
            lr
            * scale
            * gb
        )

        w -= (
            lr
            * scale
            * gW
        )

        c -= (
            lr
            * scale
            * gc
        )

    return (
        k,
        b,
        w,
        c,
    )


# ============================================================
# BASIC VISUAL HELPERS
# ============================================================

def final_patch_cells(
    grid,
    n_cols,
    top,
    left,
    size=3,
):
    """
    Select a patch from a Manim grid.
    """

    cells = []

    for r in range(
        top,
        top + size,
    ):
        for c in range(
            left,
            left + size,
        ):

            cells.append(
                grid[
                    r * n_cols
                    + c
                ]
            )

    return VGroup(
        *cells
    )


def final_probs_panel(
    probs,
    names,
):
    """
    Probability list displayed in the output block.
    """

    rows = VGroup()

    for name, p in zip(
        names,
        probs,
    ):

        row = Text(
            f"{name}: {100 * float(p):.0f}%",
            font_size=12,
        )

        rows.add(row)

    rows.arrange(
        DOWN,
        aligned_edge=LEFT,
        buff=0.06,
    )

    return rows


# ============================================================
# OLD MATRIX / VECTOR HELPERS
# Kept because they were useful in previous versions.
# ============================================================

def final_numeric_matrix(
    data,
    decimals=2,
    font_size=12,
    h_buff=0.42,
    v_buff=0.36,
):
    arr = np.asarray(
        data,
        dtype=float,
    )

    strings = [
        [
            f"{float(value):.{decimals}f}"
            for value in row
        ]
        for row in arr
    ]

    return Matrix(
        strings,
        element_to_mobject=lambda s:
            Text(
                s,
                font_size=font_size,
            ),
        h_buff=h_buff,
        v_buff=v_buff,
    )


def final_vector(
    values,
    decimals=2,
    font_size=13,
):
    arr = np.asarray(
        values,
        dtype=float,
    ).reshape(
        -1,
        1,
    )

    strings = [
        [
            f"{float(value):.{decimals}f}"
        ]
        for value in arr.flatten()
    ]

    return Matrix(
        strings,
        element_to_mobject=lambda s:
            Text(
                s,
                font_size=font_size,
            ),
        h_buff=0.35,
        v_buff=0.38,
    )


def final_block(
    title,
    body,
    width,
    height=1.35,
):
    rect = RoundedRectangle(
        width=width,
        height=height,
        corner_radius=0.10,
        stroke_color=GRAY_B,
        stroke_width=1.3,
    )

    label = Text(
        title,
        font_size=16,
    )

    label.move_to(
        rect.get_top()
        + DOWN * 0.18
    )

    body.move_to(
        rect.get_center()
        + DOWN * 0.12
    )

    return VGroup(
        rect,
        label,
        body,
    )


# ============================================================
# FEATURE MAP VISUAL HELPERS
# ============================================================

def final_signed_preview(
    data,
    cell_size=0.19,
):
    """
    Compact convolution preview.

    Positive -> blue
    Negative -> red
    """

    arr = np.asarray(
        data,
        dtype=float,
    )

    max_abs = max(
        np.max(
            np.abs(arr)
        ),
        1e-6,
    )

    cells = VGroup()

    for value in arr.flatten():

        square = Square(
            side_length=cell_size,
            stroke_color=GRAY_B,
            stroke_width=1.0,
        )

        if value > 0:

            color = interpolate_color(
                GRAY_D,
                BLUE_C,
                min(
                    abs(value)
                    / max_abs,
                    1,
                ),
            )

            square.set_fill(
                color,
                opacity=0.85,
            )

        elif value < 0:

            color = interpolate_color(
                GRAY_D,
                RED_C,
                min(
                    abs(value)
                    / max_abs,
                    1,
                ),
            )

            square.set_fill(
                color,
                opacity=0.85,
            )

        else:

            square.set_fill(
                BLACK,
                opacity=0,
            )

        cells.add(
            square
        )

    cells.arrange_in_grid(
        rows=arr.shape[0],
        cols=arr.shape[1],
        buff=0,
    )

    return cells


def final_positive_preview(
    data,
    cell_size=0.25,
    color_target=YELLOW_C,
):
    """
    Compact preview for positive-valued maps.
    """

    arr = np.asarray(
        data,
        dtype=float,
    )

    max_val = max(
        np.max(arr),
        1e-6,
    )

    cells = VGroup()

    for value in arr.flatten():

        square = Square(
            side_length=cell_size,
            stroke_color=GRAY_B,
            stroke_width=1.0,
        )

        intensity = min(
            value / max_val
            if max_val > 0
            else 0,
            1,
        )

        color = interpolate_color(
            GRAY_D,
            color_target,
            intensity,
        )

        square.set_fill(
            color,
            opacity=0.85
            if value > 0
            else 0,
        )

        cells.add(
            square
        )

    cells.arrange_in_grid(
        rows=arr.shape[0],
        cols=arr.shape[1],
        buff=0,
    )

    return cells


# ============================================================
# GRID WITH VALUES
# ============================================================

def final_value_grid(
    data,
    cell_size=0.52,
    decimals=2,
    font_size=13,
    fill=False,
    positive_color=BLUE_C,
    negative_color=RED_C,
):
    """
    Numeric grid with one value per square.

    This replaced Matrix() in the final animation
    because it gives us much more control over spacing.
    """

    arr = np.asarray(
        data,
        dtype=float,
    )

    max_abs = max(
        np.max(
            np.abs(arr)
        ),
        1e-8,
    )

    cells = VGroup()

    for value in arr.flatten():

        square = Square(
            side_length=cell_size,
            stroke_color=GRAY_B,
            stroke_width=1.2,
        )

        if fill:

            intensity = min(
                abs(value)
                / max_abs,
                1,
            )

            if value > 0:

                color = interpolate_color(
                    GRAY_E,
                    positive_color,
                    intensity,
                )

                square.set_fill(
                    color,
                    opacity=0.65,
                )

            elif value < 0:

                color = interpolate_color(
                    GRAY_E,
                    negative_color,
                    intensity,
                )

                square.set_fill(
                    color,
                    opacity=0.65,
                )

            else:

                square.set_fill(
                    BLACK,
                    opacity=0,
                )

        else:

            square.set_fill(
                BLACK,
                opacity=0,
            )

        number = Text(
            f"{value:.{decimals}f}",
            font_size=font_size,
        )

        cells.add(
            VGroup(
                square,
                number,
            )
        )

    cells.arrange_in_grid(
        rows=arr.shape[0],
        cols=arr.shape[1],
        buff=0,
    )

    return cells


# ============================================================
# OLDER DETAILED FEATURE MAP HELPERS
# ============================================================

def final_feature_map_grid(
    data,
    cell_size=0.21,
    font_size=10,
):
    arr = np.asarray(
        data,
        dtype=float,
    )

    max_abs = max(
        np.max(
            np.abs(arr)
        ),
        1e-6,
    )

    cells = VGroup()

    for value in arr.flatten():

        square = Square(
            side_length=cell_size,
            stroke_color=GRAY_B,
            stroke_width=1.0,
        )

        if value > 0:

            color = interpolate_color(
                GRAY_D,
                BLUE_C,
                min(
                    abs(value)
                    / max_abs,
                    1,
                ),
            )

            square.set_fill(
                color,
                opacity=0.85,
            )

        elif value < 0:

            color = interpolate_color(
                GRAY_D,
                RED_C,
                min(
                    abs(value)
                    / max_abs,
                    1,
                ),
            )

            square.set_fill(
                color,
                opacity=0.85,
            )

        else:

            square.set_fill(
                BLACK,
                opacity=0,
            )

        text = Text(
            f"{value:.1f}",
            font_size=font_size,
        )

        cells.add(
            VGroup(
                square,
                text,
            )
        )

    cells.arrange_in_grid(
        rows=arr.shape[0],
        cols=arr.shape[1],
        buff=0,
    )

    return cells


def final_relu_map_grid(
    data,
    cell_size=0.21,
    font_size=10,
):
    arr = np.asarray(
        data,
        dtype=float,
    )

    max_val = max(
        np.max(arr),
        1e-6,
    )

    cells = VGroup()

    for value in arr.flatten():

        square = Square(
            side_length=cell_size,
            stroke_color=GRAY_B,
            stroke_width=1.0,
        )

        intensity = (
            value / max_val
            if max_val > 0
            else 0
        )

        color = interpolate_color(
            GRAY_D,
            GREEN_C,
            min(
                max(
                    intensity,
                    0,
                ),
                1,
            ),
        )

        square.set_fill(
            color,
            opacity=0.85
            if value > 0
            else 0,
        )

        text = Text(
            f"{value:.1f}",
            font_size=font_size,
        )

        cells.add(
            VGroup(
                square,
                text,
            )
        )

    cells.arrange_in_grid(
        rows=arr.shape[0],
        cols=arr.shape[1],
        buff=0,
    )

    return cells


def final_pool_grid(
    data,
    cell_size=0.30,
    font_size=11,
):
    arr = np.asarray(
        data,
        dtype=float,
    )

    max_val = max(
        np.max(arr),
        1e-6,
    )

    cells = VGroup()

    for value in arr.flatten():

        square = Square(
            side_length=cell_size,
            stroke_color=GRAY_B,
            stroke_width=1.0,
        )

        intensity = (
            value / max_val
            if max_val > 0
            else 0
        )

        color = interpolate_color(
            GRAY_D,
            YELLOW_C,
            min(
                max(
                    intensity,
                    0,
                ),
                1,
            ),
        )

        square.set_fill(
            color,
            opacity=0.85
            if value > 0
            else 0,
        )

        text = Text(
            f"{value:.2f}",
            font_size=font_size,
        )

        cells.add(
            VGroup(
                square,
                text,
            )
        )

    cells.arrange_in_grid(
        rows=arr.shape[0],
        cols=arr.shape[1],
        buff=0,
    )

    return cells


# ============================================================
# MULTIPLICATION TEXT
# ============================================================

def final_mult_text_lines(
    patch,
    kernel,
    decimals=2,
    font_size=19,
):
    """
    Shows the arithmetic for convolution.

    Important:
    Visual multiplication symbol is now '*',
    matching the operator used in the presentation.
    """

    lines = VGroup()

    for r in range(
        patch.shape[0]
    ):

        expression = " + ".join(
            [
                f"{int(patch[r, c])} * {kernel[r, c]:.{decimals}f}"
                for c in range(
                    patch.shape[1]
                )
            ]
        )

        lines.add(
            Text(
                expression,
                font_size=font_size,
            )
        )

    lines.arrange(
        DOWN,
        aligned_edge=LEFT,
        buff=0.06,
    )

    return lines


# Older alias, in case an older scene still uses this name.
def final_mult_lines(
    patch,
    kernel,
    decimals=2,
    font_size=18,
):
    return final_mult_text_lines(
        patch,
        kernel,
        decimals=decimals,
        font_size=font_size,
    )

In [ ]:
# ============================================================
# FINAL VIDEO — CLEANER LAYOUT
# Replace the previous VideoFinalCompleteCNN class.
# ============================================================

class VideoFinalCompleteCNN(Scene):

    def construct(self):

        # ----------------------------------------------------
        # Data
        # ----------------------------------------------------

        symbols = make_symbol_images()

        plus = symbols["+"].astype(float)
        minus = symbols["−"].astype(float)
        slash = symbols["/"].astype(float)
        times = symbols['*'].astype(float)

        samples = [plus, minus, slash, times]
        names = ["+", "−", "/", "*"]
        labels = [0, 1, 2, 3]

        # Fixed initialization
        K0 = np.array([
            [ 0.42, -0.18,  0.11],
            [-0.33,  0.29,  0.37],
            [ 0.15, -0.24,  0.21],
        ])

        b0 = -0.12

        W0 = np.array([
            [ 0.25, -0.18,  0.10,  0.06],
            [-0.15,  0.24, -0.07,  0.05],
            [ 0.08, -0.05,  0.21, -0.14],
            [-0.07,  0.06, -0.12,  0.22],
        ])

        c0 = np.zeros(4)

        f0, delta, dK, db, dW, dc, dZ = final_backward(
            plus, 0, K0, b0, W0, c0
        )

        eta = 0.60

        K1 = K0 - eta * dK
        b1 = b0 - eta * db
        W1 = W0 - eta * dW
        c1 = c0 - eta * dc

        f1 = final_forward(plus, K1, b1, W1, c1)

        Kt, bt, Wt, ct = final_train(
            samples, labels, K1, b1, W1, c1,
            epochs=800, lr=0.08
        )

        # ----------------------------------------------------
        # Local helpers
        # ----------------------------------------------------

        def stage_box(
            title,
            center,
            width,
            height=1.95,
            label_font_size=17,
        ):
            rect = RoundedRectangle(
                width=width,
                height=height,
                corner_radius=0.08,
                stroke_color=GRAY_B,
                stroke_width=1.35,
            )

            rect.move_to(center)

            label = Text(
                title,
                font_size=label_font_size,
            )

            label.move_to(
                rect.get_top()
                + DOWN * 0.18
            )

            return rect, label

        def focus(rect, old=None, color=YELLOW):
            new = SurroundingRectangle(
                rect,
                buff=0.05,
                color=color,
                stroke_width=4,
            )
            new.set_z_index(20)

            if old is None:
                self.play(Create(new), run_time=0.30)
            else:
                self.play(ReplacementTransform(old, new), run_time=0.30)

            self.play(
                new.animate.scale(1.04),
                rate_func=there_and_back,
                run_time=0.28,
            )
            return new

        def clear_detail(*objs):
            valid = [o for o in objs if o is not None]
            if valid:
                self.play(*[FadeOut(o) for o in valid], run_time=0.45)

        def forward_pulse(arrows, color=GREEN_C, run_time=0.14):
            pulse = Dot(radius=0.05, color=color)
            self.add(pulse)

            for a in arrows:
                pulse.move_to(a.get_start())
                self.play(MoveAlongPath(pulse, a), run_time=run_time)

            self.remove(pulse)

        # ----------------------------------------------------
        # Top pipeline
        # ----------------------------------------------------

        y_top = 2.35

        input_rect, input_label = stage_box(
            "Input",
            [-5.45, y_top, 0],
            1.90,
        )

        conv_rect, conv_label = stage_box(
            "Convolution + Bias",
            [-3.15, y_top, 0],
            2.50,
            label_font_size=16,
        )

        relu_rect, relu_label = stage_box(
            "ReLU",
            [-0.65, y_top, 0],
            1.40,
        )

        pool_rect, pool_label = stage_box(
            "Pooling",
            [1.00, y_top, 0],
            1.40,
        )

        classifier_rect, classifier_label = stage_box(
            "Classifier",
            [2.90, y_top, 0],
            1.60,
        )

        output_rect, output_label = stage_box(
            "Output",
            [5.00, y_top, 0],
            1.60,
        )

        stage_rects = [
            input_rect, conv_rect, relu_rect,
            pool_rect, classifier_rect, output_rect
        ]

        stage_labels = [
            input_label, conv_label, relu_label,
            pool_label, classifier_label, output_label
        ]

        # Input preview
        input_symbol = Text("+", font_size=34)
        input_preview_grid = make_binary_grid(
            plus.astype(int),
            cell_size=0.11,
            show_values=False,
            font_size=6,
        )
        input_preview_arrow = Arrow(
            LEFT * 0.22, RIGHT * 0.22, buff=0, stroke_width=2
        )
        input_body = VGroup(
            input_symbol, input_preview_arrow, input_preview_grid
        ).arrange(RIGHT, buff=0.10)
        input_body.move_to(input_rect.get_center() + DOWN * 0.10)

        # Convolution preview: NO numbers
        conv_body = final_signed_preview(f0["z"], cell_size=0.16)
        conv_body.move_to(conv_rect.get_center() + DOWN * 0.10)

        # ReLU preview: curve only
        relu_axes = Axes(
            x_range=[-1, 1, 1],
            y_range=[0, 1, 1],
            x_length=1.0,
            y_length=0.7,
            tips=False,
            axis_config={"stroke_width": 1.2, "include_numbers": False},
        )
        relu_curve = relu_axes.plot(
            lambda x: max(0, x),
            x_range=[-1, 1],
            color=GREEN_C,
            stroke_width=2.8,
        )
        relu_body = VGroup(relu_axes, relu_curve)
        relu_body.move_to(relu_rect.get_center() + DOWN * 0.12)

        # Pooling preview: NO numbers
        pool_body = final_positive_preview(f0["pool"], cell_size=0.30, color_target=YELLOW_C)
        pool_body.move_to(pool_rect.get_center() + DOWN * 0.10)

        # Classifier preview
        classifier_body = VGroup(
            Text("Flatten", font_size=13),
            Text("↓", font_size=15),
            Text("Dense", font_size=13),
            Text("↓", font_size=15),
            Text("Softmax", font_size=13),
        ).arrange(DOWN, buff=0.02)
        classifier_body.move_to(classifier_rect.get_center() + DOWN * 0.10)

        # Output preview
        output_body = final_probs_panel(f0["probs"], names)
        output_body.scale(0.95)
        output_body.move_to(output_rect.get_center() + DOWN * 0.10)

        top_bodies = [
            input_body, conv_body, relu_body,
            pool_body, classifier_body, output_body
        ]

        self.play(
            LaggedStart(
                *[FadeIn(o) for o in stage_rects + stage_labels + top_bodies],
                lag_ratio=0.02,
            ),
            run_time=1.2,
        )

        pipeline_arrows = VGroup(*[
            Arrow(a.get_right(), b.get_left(), buff=0.07, stroke_width=2.0)
            for a, b in zip(stage_rects[:-1], stage_rects[1:])
        ])

        self.play(
            LaggedStart(*[GrowArrow(a) for a in pipeline_arrows], lag_ratio=0.07),
            run_time=0.9,
        )

        focus_rect = None

        # ====================================================
        # PHASE 1 — INPUT REPRESENTATION
        # ====================================================

        focus_rect = focus(input_rect, focus_rect)

        phase_title = Text("Image Representation", font_size=22, color=YELLOW)
        phase_title.move_to([0, -0.15, 0])

        symbol_big = Text("+", font_size=82)
        symbol_big.move_to([-4.0, -1.55, 0])

        arrow_big = Arrow([-2.8, -1.55, 0], [-1.5, -1.55, 0], buff=0.08, stroke_width=3)

        matrix_big = make_binary_grid(
            plus.astype(int),
            cell_size=0.26,
            show_values=True,
            font_size=14,
        )
        matrix_big.move_to([0.4, -1.55, 0])

        matrix_label = Text("7 × 7 binary matrix", font_size=18)
        matrix_label.next_to(matrix_big, DOWN, buff=0.14)

        self.play(FadeIn(phase_title), FadeIn(symbol_big))
        self.play(
            GrowArrow(arrow_big),
            TransformFromCopy(input_preview_grid, matrix_big),
            FadeIn(matrix_label),
            run_time=0.9,
        )
        self.wait(0.7)

        clear_detail(phase_title, symbol_big, arrow_big, matrix_big, matrix_label)

        # ====================================================
        # PHASE 2 — CONVOLUTION
        # ====================================================

        focus_rect = focus(
            conv_rect,
            focus_rect,
        )

        conv_title = Text(
            "Convolution",
            font_size=22,
            color=YELLOW,
        )

        conv_title.move_to(
            [0, -0.15, 0]
        )

        # ----------------------------------------------------
        # Full input X
        # ----------------------------------------------------

        X_view = make_binary_grid(
            plus.astype(int),
            cell_size=0.20,
            show_values=True,
            font_size=11,
        )

        X_view.move_to(
            [-4.75, -1.45, 0]
        )

        X_label = Text(
            "X",
            font_size=19,
        ).next_to(
            X_view,
            UP,
            buff=0.08,
        )

        # ----------------------------------------------------
        # Kernel K — use a real grid instead of Matrix()
        # ----------------------------------------------------

        K_view = final_value_grid(
            K0,
            cell_size=0.57,
            decimals=2,
            font_size=14,
            fill=False,
        )

        K_view.move_to(
            [-1.55, -1.45, 0]
        )

        K_label = Text(
            "K",
            font_size=19,
        ).next_to(
            K_view,
            UP,
            buff=0.08,
        )

        star = Text(
            "∗",
            font_size=30,
        )

        star.move_to(
            [-3.0, -1.45, 0]
        )

        # ----------------------------------------------------
        # Output feature map Y
        # ----------------------------------------------------

        Y_view = final_value_grid(
            f0["z"],
            cell_size=0.28,
            decimals=1,
            font_size=9,
            fill=True,
        )

        Y_view.move_to(
            [3.70, -1.45, 0]
        )

        Y_label = Text(
            "Feature Map Y",
            font_size=18,
        ).next_to(
            Y_view,
            UP,
            buff=0.08,
        )

        conv_arrow = Arrow(
            [0.25, -1.45, 0],
            [2.70, -1.45, 0],
            buff=0.15,
            stroke_width=3,
        )

        self.play(
            FadeIn(conv_title),
            FadeIn(X_view),
            FadeIn(X_label),
            FadeIn(star),
            FadeIn(K_view),
            FadeIn(K_label),
            GrowArrow(conv_arrow),
            FadeIn(Y_view),
            FadeIn(Y_label),
        )

        # ----------------------------------------------------
        # Select one patch
        # ----------------------------------------------------

        patch_top = 0
        patch_left = 2

        patch = plus[
            patch_top:patch_top + 3,
            patch_left:patch_left + 3,
        ]

        patch_cells = final_patch_cells(
            X_view,
            7,
            patch_top,
            patch_left,
            3,
        )

        patch_box = SurroundingRectangle(
            patch_cells,
            color=YELLOW,
            buff=0.015,
            stroke_width=3,
        )

        target_cell = Y_view[
            patch_top * 5
            + patch_left
        ]

        target_box = SurroundingRectangle(
            target_cell,
            color=YELLOW,
            buff=0.015,
            stroke_width=3,
        )

        self.play(
            Create(patch_box),
            Create(target_box),
        )

        # ----------------------------------------------------
        # Show actual arithmetic below
        # ----------------------------------------------------

        products = patch * K0

        row_lines = VGroup()

        for r in range(3):

            expression = " + ".join(
                [
                    f"{int(patch[r, c])} * {K0[r, c]:.2f}"
                    for c in range(3)
                ]
            )

            row_lines.add(
                Text(
                    expression,
                    font_size=17,
                )
            )

        row_lines.arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.06,
        )

        row_lines.move_to(
            [-1.1, -3.05, 0]
        )

        sum_value = float(
            np.sum(products)
        )

        output_value = float(
            f0["z"][
                patch_top,
                patch_left,
            ]
        )

        result_lines = VGroup(
            Text(
                f"sum = {sum_value:.2f}",
                font_size=18,
            ),
            Text(
                f"+ bias = {b0:.2f}",
                font_size=18,
            ),
            Text(
                f"Y[0,2] = {output_value:.2f}",
                font_size=21,
                color=GREEN_C
                if output_value >= 0
                else RED_C,
            ),
        )

        result_lines.arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.08,
        )

        result_lines.move_to(
            [3.25, -3.05, 0]
        )

        self.play(
            LaggedStart(
                *[
                    Write(line)
                    for line in row_lines
                ],
                lag_ratio=0.20,
            ),
            run_time=1.2,
        )

        self.play(
            FadeIn(result_lines)
        )

        self.play(
            Indicate(
                target_cell,
                color=YELLOW,
            )
        )

        self.wait(0.7)

        clear_detail(
            conv_title,
            X_view,
            X_label,
            K_view,
            K_label,
            star,
            conv_arrow,
            Y_view,
            Y_label,
            patch_box,
            target_box,
            row_lines,
            result_lines,
        )

        # ====================================================
        # PHASE 3 — RELU
        # ====================================================

        focus_rect = focus(relu_rect, focus_rect)

        relu_title = Text("ReLU Activation", font_size=22, color=YELLOW)
        relu_title.move_to([0, -0.15, 0])

        neg_positions = np.argwhere(f0["z"] < 0)
        if len(neg_positions) > 0:
            rr, cc = neg_positions[0]
        else:
            rr, cc = 0, 0

        z_val = float(f0["z"][rr, cc])
        a_val = max(0.0, z_val)

        axes = Axes(
            x_range=[-1.5, 1.5, 0.5],
            y_range=[0, 1.5, 0.5],
            x_length=4.8,
            y_length=2.3,
            axis_config={"include_numbers": True, "font_size": 15, "stroke_width": 1.4},
        )
        axes.move_to([-1.0, -1.55, 0])

        curve = axes.plot(
            lambda x: max(0, x),
            x_range=[-1.5, 1.5],
            color=GREEN_C,
            stroke_width=3.6,
        )

        point = Dot(axes.c2p(z_val, a_val), radius=0.06, color=YELLOW)

        z_text = Text(f"Input z = {z_val:.2f}", font_size=20)
        z_text.move_to([3.85, -1.20, 0])

        relu_text = Text(f"ReLU(z) = {a_val:.2f}", font_size=22, color=GREEN_C)
        relu_text.next_to(z_text, DOWN, buff=0.20)

        self.play(FadeIn(relu_title), Create(axes), Create(curve))
        self.play(FadeIn(z_text))
        self.play(FadeIn(point), FadeIn(relu_text), run_time=0.6)

        self.wait(0.85)

        clear_detail(relu_title, axes, curve, point, z_text, relu_text)

        # ====================================================
        # PHASE 4 — POOLING
        # ====================================================

        focus_rect = focus(
            pool_rect,
            focus_rect,
        )

        pool_title = Text(
            "Pooling",
            font_size=22,
            color=YELLOW,
        )

        pool_title.move_to(
            [0, -0.15, 0]
        )

        # ----------------------------------------------------
        # Larger feature map
        # ----------------------------------------------------

        relu_big = final_value_grid(
            f0["a"],
            cell_size=0.38,
            decimals=2,
            font_size=11,
            fill=True,
            positive_color=GREEN_C,
        )

        relu_big.move_to(
            [-4.15, -1.55, 0]
        )

        relu_label = Text(
            "Feature Map",
            font_size=18,
        ).next_to(
            relu_big,
            UP,
            buff=0.08,
        )

        # ----------------------------------------------------
        # First pooling window
        # ----------------------------------------------------

        selected_cells = VGroup(
            relu_big[0],
            relu_big[1],
            relu_big[5],
            relu_big[6],
        )

        pool_highlight = SurroundingRectangle(
            selected_cells,
            color=YELLOW,
            buff=0.015,
            stroke_width=3,
        )

        self.play(
            FadeIn(pool_title),
            FadeIn(relu_big),
            FadeIn(relu_label),
        )

        self.play(
            Create(pool_highlight)
        )

        # ----------------------------------------------------
        # Formula with more vertical space
        # ----------------------------------------------------

        pool_values = [
            f0["a"][0, 0],
            f0["a"][0, 1],
            f0["a"][1, 0],
            f0["a"][1, 1],
        ]

        pool_formula = VGroup(
            Text(
                f"{pool_values[0]:.2f} + {pool_values[1]:.2f}",
                font_size=20,
            ),
            Text(
                f"+ {pool_values[2]:.2f} + {pool_values[3]:.2f}",
                font_size=20,
            ),
            Text(
                "────────────",
                font_size=18,
            ),
            Text(
                "4",
                font_size=20,
            ),
            Text(
                f"= {f0['pool'][0,0]:.2f}",
                font_size=23,
                color=YELLOW_C,
            ),
        )

        pool_formula.arrange(
            DOWN,
            buff=0.03,
        )

        pool_formula.move_to(
            [0.0, -1.55, 0]
        )

        # ----------------------------------------------------
        # Resulting 2x2 map
        # ----------------------------------------------------

        pool_arrow = Arrow(
            [1.60, -1.55, 0],
            [3.05, -1.55, 0],
            buff=0.10,
            stroke_width=3,
        )

        pooled_big = final_value_grid(
            f0["pool"],
            cell_size=0.67,
            decimals=2,
            font_size=15,
            fill=True,
            positive_color=YELLOW_C,
        )

        pooled_big.move_to(
            [4.45, -1.55, 0]
        )

        pooled_label = Text(
            "Pooled Feature Map",
            font_size=18,
        ).next_to(
            pooled_big,
            UP,
            buff=0.10,
        )

        self.play(
            FadeIn(pool_formula)
        )

        self.play(
            GrowArrow(pool_arrow),
            FadeIn(pooled_big),
            FadeIn(pooled_label),
            run_time=0.7,
        )

        self.wait(0.8)

        clear_detail(
            pool_title,
            relu_big,
            relu_label,
            pool_highlight,
            pool_formula,
            pool_arrow,
            pooled_big,
            pooled_label,
        )

        # ====================================================
        # PHASE 5 — CLASSIFIER
        # ====================================================

        focus_rect = focus(
            classifier_rect,
            focus_rect,
        )

        class_title = Text(
            "Classification",
            font_size=22,
            color=YELLOW,
        )

        class_title.move_to(
            [0, -0.15, 0]
        )

        # ----------------------------------------------------
        # Pooling result
        # ----------------------------------------------------

        pool_matrix = final_value_grid(
            f0["pool"],
            cell_size=0.60,
            decimals=2,
            font_size=14,
            fill=True,
            positive_color=YELLOW_C,
        )

        pool_matrix.move_to(
            [-4.55, -1.55, 0]
        )

        pool_matrix_label = Text(
            "2 × 2",
            font_size=17,
        ).next_to(
            pool_matrix,
            UP,
            buff=0.08,
        )

        # ----------------------------------------------------
        # Flatten vector
        # ----------------------------------------------------

        flat_grid = final_value_grid(
            f0["h"].reshape(4, 1),
            cell_size=0.50,
            decimals=2,
            font_size=13,
            fill=False,
        )

        flat_grid.move_to(
            [-1.65, -1.55, 0]
        )

        flat_label = Text(
            "Flatten",
            font_size=18,
        ).next_to(
            flat_grid,
            UP,
            buff=0.08,
        )

        flatten_arrow = Arrow(
            [-3.50, -1.55, 0],
            [-2.45, -1.55, 0],
            buff=0.08,
            stroke_width=3,
        )

        self.play(
            FadeIn(class_title),
            FadeIn(pool_matrix),
            FadeIn(pool_matrix_label),
        )

        self.play(
            GrowArrow(flatten_arrow)
        )

        # Send every pooling cell into its vector position
        for i in range(4):

            self.play(
                TransformFromCopy(
                    pool_matrix[i],
                    flat_grid[i],
                ),
                run_time=0.25,
            )

        self.play(
            FadeIn(flat_label)
        )

        # ----------------------------------------------------
        # Dense
        # ----------------------------------------------------

        dense_arrow = Arrow(
            [-0.80, -1.55, 0],
            [0.35, -1.55, 0],
            buff=0.08,
            stroke_width=3,
        )

        dense_box = RoundedRectangle(
            width=1.55,
            height=1.15,
            corner_radius=0.08,
            stroke_color=GRAY_B,
        )

        dense_box.move_to(
            [1.30, -1.55, 0]
        )

        dense_label = Text(
            "Dense",
            font_size=20,
        )

        dense_label.move_to(
            dense_box.get_center()
        )

        softmax_arrow = Arrow(
            [2.15, -1.55, 0],
            [3.25, -1.55, 0],
            buff=0.08,
            stroke_width=3,
        )

        probabilities = final_probs_panel(
            f0["probs"],
            names,
        )

        probabilities.scale(1.3)

        probabilities.move_to(
            [4.65, -1.55, 0]
        )

        self.play(
            GrowArrow(dense_arrow),
            FadeIn(dense_box),
            FadeIn(dense_label),
        )

        self.play(
            GrowArrow(softmax_arrow),
            FadeIn(probabilities),
        )

        self.wait(0.8)

        clear_detail(
            class_title,
            pool_matrix,
            pool_matrix_label,
            flatten_arrow,
            flat_grid,
            flat_label,
            dense_arrow,
            dense_box,
            dense_label,
            softmax_arrow,
            probabilities,
        )

        # ====================================================
        # PHASE 6 — BACKPROPAGATION
        # ====================================================

        focus_rect = focus(
            output_rect,
            focus_rect,
            color=RED_C,
        )

        back_title = Text(
            "Backpropagation",
            font_size=23,
            color=RED_C,
        )

        back_title.move_to(
            [0, -0.15, 0]
        )

        backward_arrows = VGroup(
            *[
                Arrow(
                    b.get_left(),
                    a.get_right(),
                    buff=0.08,
                    stroke_width=2.4,
                    color=RED_C,
                )
                for a, b
                in zip(
                    stage_rects[:-1],
                    stage_rects[1:],
                )
            ]
        )

        self.play(
            FadeIn(back_title)
        )

        self.play(
            LaggedStart(
                *[
                    GrowArrow(a)
                    for a
                    in backward_arrows[::-1]
                ],
                lag_ratio=0.08,
            ),
            run_time=1.0,
        )

        # ----------------------------------------------------
        # Pick one non-zero dZ
        # ----------------------------------------------------

        positions = np.argwhere(
            np.abs(dZ) > 1e-8
        )

        if len(positions) > 0:
            gr, gc_ = positions[0]
        else:
            gr, gc_ = 0, 0

        grad_value = float(
            dZ[gr, gc_]
        )

        source_patch = plus[
            gr:gr + 3,
            gc_:gc_ + 3,
        ]

        # ====================================================
        # STEP 1 — Gradient flowing toward X
        # ====================================================

        step1 = Text(
            "1. Propagate the gradient through the convolution",
            font_size=19,
            color=RED_C,
        )

        step1.move_to(
            [0, -0.55, 0]
        )

        self.play(
            FadeIn(step1)
        )

        dz_grid = final_value_grid(
            dZ,
            cell_size=0.35,
            decimals=2,
            font_size=9,
            fill=False,
        )

        dz_grid.move_to(
            [-4.90, -2.15, 0]
        )

        dz_label = Text(
            "dZ",
            font_size=18,
        ).next_to(
            dz_grid,
            UP,
            buff=0.08,
        )

        selected_index = (
            gr * dZ.shape[1]
            + gc_
        )

        selected_cell = dz_grid[
            selected_index
        ]

        selected_box = SurroundingRectangle(
            selected_cell,
            color=YELLOW,
            buff=0.02,
            stroke_width=3,
        )

        self.play(
            FadeIn(dz_grid),
            FadeIn(dz_label),
            Create(selected_box),
        )

        grad_scalar = Text(
            f"{grad_value:.2f}",
            font_size=25,
            color=RED_C,
        )

        grad_scalar.move_to(
            [-2.80, -2.15, 0]
        )

        self.play(
            TransformFromCopy(
                selected_cell,
                grad_scalar,
            )
        )

        # ----------------------------------------------------
        # K comes from the forward convolution
        # ----------------------------------------------------

        kernel_back = final_value_grid(
            K0,
            cell_size=0.48,
            decimals=2,
            font_size=12,
            fill=False,
        )

        kernel_back.move_to(
            [-0.75, -2.15, 0]
        )

        kernel_back_label = Text(
            "K used in the forward pass",
            font_size=16,
        ).next_to(
            kernel_back,
            UP,
            buff=0.08,
        )

        times_sign = Text(
            "*",
            font_size=30,
        )

        times_sign.move_to(
            [-1.90, -2.15, 0]
        )

        self.play(
            FadeIn(times_sign),
            FadeIn(kernel_back),
            FadeIn(kernel_back_label),
        )

        # ----------------------------------------------------
        # Contribution to dX
        # ----------------------------------------------------

        dx_contribution = (
            grad_value
            * K0
        )

        dx_grid = final_value_grid(
            dx_contribution,
            cell_size=0.48,
            decimals=2,
            font_size=12,
            fill=True,
        )

        dx_grid.move_to(
            [2.15, -2.15, 0]
        )

        equals_dx = Text(
            "=",
            font_size=30,
        )

        equals_dx.move_to(
            [0.65, -2.15, 0]
        )

        dx_label = Text(
            "Contribution to dX",
            font_size=16,
        ).next_to(
            dx_grid,
            UP,
            buff=0.08,
        )

        self.play(
            FadeIn(equals_dx),
            FadeIn(dx_grid),
            FadeIn(dx_label),
        )

        self.wait(0.8)

        clear_detail(
            step1,
            dz_grid,
            dz_label,
            selected_box,
            grad_scalar,
            times_sign,
            kernel_back,
            kernel_back_label,
            equals_dx,
            dx_grid,
            dx_label,
        )

        # ====================================================
        # STEP 2 — Gradient with respect to K
        # ====================================================

        step2 = Text(
            "2. Compute the gradient of the kernel",
            font_size=19,
            color=RED_C,
        )

        step2.move_to(
            [0, -0.55, 0]
        )

        self.play(
            FadeIn(step2)
        )

        grad_scalar2 = Text(
            f"{grad_value:.2f}",
            font_size=25,
            color=RED_C,
        )

        grad_scalar2.move_to(
            [-4.80, -2.10, 0]
        )

        times2 = Text(
            "*",
            font_size=30,
        )

        times2.move_to(
            [-3.80, -2.10, 0]
        )

        # ----------------------------------------------------
        # Show where the patch comes from
        # ----------------------------------------------------

        full_X = make_binary_grid(
            plus.astype(int),
            cell_size=0.17,
            show_values=True,
            font_size=9,
        )

        full_X.move_to(
            [-2.30, -2.10, 0]
        )

        X_label = Text(
            "X",
            font_size=17,
        ).next_to(
            full_X,
            UP,
            buff=0.07,
        )

        patch_cells = final_patch_cells(
            full_X,
            7,
            gr,
            gc_,
            3,
        )

        patch_highlight = SurroundingRectangle(
            patch_cells,
            color=YELLOW,
            buff=0.01,
            stroke_width=3,
        )

        self.play(
            FadeIn(grad_scalar2),
            FadeIn(times2),
            FadeIn(full_X),
            FadeIn(X_label),
            Create(patch_highlight),
        )

        # ----------------------------------------------------
        # Extract patch
        # ----------------------------------------------------

        patch_copy = make_binary_grid(
            source_patch.astype(int),
            cell_size=0.35,
            show_values=True,
            font_size=14,
        )

        patch_copy.move_to(
            [0.15, -2.10, 0]
        )

        self.play(
            TransformFromCopy(
                patch_cells,
                patch_copy,
            )
        )

        # ----------------------------------------------------
        # One contribution to dK
        # ----------------------------------------------------

        contribution1 = (
            grad_value
            * source_patch
        )

        contribution1_grid = final_value_grid(
            contribution1,
            cell_size=0.40,
            decimals=2,
            font_size=11,
            fill=True,
            positive_color=RED_C,
            negative_color=BLUE_C,
        )

        contribution1_grid.move_to(
            [2.20, -2.10, 0]
        )

        equals_dk = Text(
            "=",
            font_size=30,
        )

        equals_dk.move_to(
            [1.15, -2.10, 0]
        )

        C1_label = Text(
            "Contribution 1",
            font_size=15,
        ).next_to(
            contribution1_grid,
            UP,
            buff=0.07,
        )

        self.play(
            FadeIn(equals_dk),
            FadeIn(contribution1_grid),
            FadeIn(C1_label),
        )

        self.wait(0.55)

        # ----------------------------------------------------
        # Show that other spatial locations make other
        # contributions.
        # ----------------------------------------------------

        other_contributions = []

        for pos in positions[1:3]:

            r, c = pos

            contribution = (
                dZ[r, c]
                * plus[
                    r:r + 3,
                    c:c + 3,
                ]
            )

            grid = final_value_grid(
                contribution,
                cell_size=0.27,
                decimals=2,
                font_size=8,
                fill=False,
            )

            other_contributions.append(
                grid
            )

        if len(other_contributions) > 0:

            plus_sign1 = Text(
                "+",
                font_size=26,
            )

            small_group = VGroup()

            small_group.add(
                contribution1_grid.copy().scale(0.68)
            )

            for contribution in other_contributions:

                small_group.add(
                    Text(
                        "+",
                        font_size=24,
                    )
                )

                small_group.add(
                    contribution
                )

            small_group.add(
                Text(
                    "+ ...",
                    font_size=22,
                )
            )

            small_group.arrange(
                RIGHT,
                buff=0.12,
            )

            small_group.move_to(
                [1.80, -3.20, 0]
            )

            self.play(
                FadeIn(small_group)
            )

        else:

            small_group = None

        # ----------------------------------------------------
        # Final dK
        # ----------------------------------------------------

        dk_arrow = Arrow(
            [4.00, -3.20, 0],
            [4.75, -3.20, 0],
            buff=0.05,
            stroke_width=3,
        )

        dK_final = final_value_grid(
            dK,
            cell_size=0.40,
            decimals=2,
            font_size=11,
            fill=False,
        )

        dK_final.move_to(
            [5.55, -3.20, 0]
        )

        dK_final_label = Text(
            "∇K L",
            font_size=17,
            color=RED_C,
        ).next_to(
            dK_final,
            UP,
            buff=0.07,
        )

        self.play(
            GrowArrow(dk_arrow),
            FadeIn(dK_final),
            FadeIn(dK_final_label),
        )

        self.wait(0.8)

        # ====================================================
        # STEP 3 — UPDATE K
        # ====================================================

        clear_detail(
            step2,
            grad_scalar2,
            times2,
            full_X,
            X_label,
            patch_highlight,
            patch_copy,
            equals_dk,
            contribution1_grid,
            C1_label,
            small_group,
            dk_arrow,
        )

        step3 = Text(
            "3. Update the kernel",
            font_size=19,
            color=RED_C,
        )

        step3.move_to(
            [0, -0.55, 0]
        )

        self.play(
            FadeIn(step3)
        )

        K_old = final_value_grid(
            K0,
            cell_size=0.45,
            decimals=2,
            font_size=11,
        )

        eta_dK = final_value_grid(
            eta * dK,
            cell_size=0.45,
            decimals=2,
            font_size=11,
        )

        K_new = final_value_grid(
            K1,
            cell_size=0.45,
            decimals=2,
            font_size=11,
        )

        K_old.move_to(
            [-3.70, -2.20, 0]
        )

        eta_dK.move_to(
            [-0.55, -2.20, 0]
        )

        K_new.move_to(
            [3.20, -2.20, 0]
        )

        old_label = Text(
            "K",
            font_size=17,
        ).next_to(
            K_old,
            UP,
            buff=0.08,
        )

        gradient_label = Text(
            "η ∇K L",
            font_size=17,
            color=RED_C,
        ).next_to(
            eta_dK,
            UP,
            buff=0.08,
        )

        new_label = Text(
            "Updated K",
            font_size=17,
            color=GREEN_C,
        ).next_to(
            K_new,
            UP,
            buff=0.08,
        )

        minus_sign = Text(
            "−",
            font_size=32,
        )

        minus_sign.move_to(
            [-2.10, -2.20, 0]
        )

        equals_sign = Text(
            "=",
            font_size=30,
        )

        equals_sign.move_to(
            [1.20, -2.20, 0]
        )

        self.play(
            FadeIn(K_old),
            FadeIn(old_label),
        )

        # Move dK into ηdK, then remove the original.
        self.play(
            TransformFromCopy(
                dK_final,
                eta_dK,
            ),
            FadeIn(minus_sign),
            FadeIn(gradient_label),
        )

        self.play(
            FadeOut(dK_final),
            FadeOut(dK_final_label),
        )

        self.play(
            FadeIn(equals_sign),
            FadeIn(K_new),
            FadeIn(new_label),
        )

        self.wait(0.7)

        # ----------------------------------------------------
        # Send the updated K back to convolution
        # ----------------------------------------------------

        flying_K = K_new.copy()

        self.play(
            flying_K.animate
            .scale(0.28)
            .move_to(
                conv_rect.get_center()
            ),
            run_time=0.9,
        )

        self.play(
            Flash(
                conv_rect.get_center(),
                color=GREEN_C,
                flash_radius=1.0,
            ),
            FadeOut(flying_K),
        )

        clear_detail(
            step3,
            K_old,
            old_label,
            minus_sign,
            eta_dK,
            gradient_label,
            equals_sign,
            K_new,
            new_label,
        )

        self.play(
            FadeOut(back_title),
            FadeOut(backward_arrows),
        )

        # ====================================================
        # FINAL — DIFFERENT SYMBOLS, SLOWER AND CLEARER
        # ====================================================

        final_text = Text("After repeated updates", font_size=21, color=GREEN_C)
        final_text.move_to([0, -0.35, 0])

        self.play(FadeIn(final_text))

        for sample, name in zip(
            samples,
            names,
        ):

            pred = final_forward(
                sample,
                Kt,
                bt,
                Wt,
                ct,
            )

            # ------------------------------------------------
            # 1. Change input
            # ------------------------------------------------

            new_input_symbol = Text(
                name,
                font_size=34,
            )

            new_input_grid = make_binary_grid(
                sample.astype(int),
                cell_size=0.11,
                show_values=False,
                font_size=6,
            )

            new_input_arrow = Arrow(
                LEFT * 0.22,
                RIGHT * 0.22,
                buff=0,
                stroke_width=2,
            )

            new_input_body = VGroup(
                new_input_symbol,
                new_input_arrow,
                new_input_grid,
            ).arrange(
                RIGHT,
                buff=0.10,
            )

            new_input_body.move_to(
                input_body.get_center()
            )

            self.play(
                Transform(
                    input_body,
                    new_input_body,
                ),
                run_time=0.55,
            )

            focus_rect = focus(
                input_rect,
                focus_rect,
                color=GREEN_C,
            )

            self.wait(0.25)

            # ------------------------------------------------
            # 2. Convolution
            # ------------------------------------------------

            new_conv = final_signed_preview(
                pred["z"],
                cell_size=0.16,
            )

            new_conv.move_to(
                conv_body.get_center()
            )

            focus_rect = focus(
                conv_rect,
                focus_rect,
                color=GREEN_C,
            )

            self.play(
                Transform(
                    conv_body,
                    new_conv,
                ),
                run_time=0.45,
            )

            self.wait(0.15)

            # ------------------------------------------------
            # 3. ReLU
            # ------------------------------------------------

            focus_rect = focus(
                relu_rect,
                focus_rect,
                color=GREEN_C,
            )

            self.play(
                Indicate(
                    relu_body,
                    color=GREEN_C,
                ),
                run_time=0.35,
            )

            # ------------------------------------------------
            # 4. Pooling
            # ------------------------------------------------

            new_pool = final_positive_preview(
                pred["pool"],
                cell_size=0.30,
                color_target=YELLOW_C,
            )

            new_pool.move_to(
                pool_body.get_center()
            )

            focus_rect = focus(
                pool_rect,
                focus_rect,
                color=GREEN_C,
            )

            self.play(
                Transform(
                    pool_body,
                    new_pool,
                ),
                run_time=0.40,
            )

            # ------------------------------------------------
            # 5. Classifier
            # ------------------------------------------------

            focus_rect = focus(
                classifier_rect,
                focus_rect,
                color=GREEN_C,
            )

            self.play(
                Indicate(
                    classifier_body,
                    color=GREEN_C,
                ),
                run_time=0.35,
            )

            # ------------------------------------------------
            # 6. Output
            # ------------------------------------------------

            new_output = final_probs_panel(
                pred["probs"],
                names,
            )

            new_output.scale(
                0.95
            )

            new_output.move_to(
                output_body.get_center()
            )

            self.play(
                Transform(
                    output_body,
                    new_output,
                ),
                run_time=0.45,
            )

            focus_rect = focus(
                output_rect,
                focus_rect,
                color=GREEN_C,
            )

            prediction = names[
                int(
                    np.argmax(
                        pred["probs"]
                    )
                )
            ]

            prediction_text = Text(
                f"Prediction: {prediction}",
                font_size=24,
                color=GREEN_C,
            )

            prediction_text.move_to(
                [0, -1.2, 0]
            )

            self.play(
                FadeIn(
                    prediction_text
                )
            )

            self.wait(0.8)

            self.play(
                FadeOut(
                    prediction_text
                )
            )

        self.play(FadeOut(final_text))
        self.wait(1.0)

In [ ]:
%manim -ql -v ERROR --progress_bar none Video1ImageRepresentation

Manim Community v0.21.0

In [ ]:
%manim -ql -v ERROR --progress_bar none Video2ConvolutionInnerProduct

Manim Community v0.21.0

In [ ]:
%manim -ql -v ERROR --progress_bar none Video3MatrixRepresentation

Manim Community v0.21.0

In [ ]:
%manim -ql -v ERROR --progress_bar none Video4Im2colGemm

Manim Community v0.21.0

In [ ]:
%manim -ql -v ERROR --progress_bar none Video5BackpropTranspose

Manim Community v0.21.0

In [ ]:
%manim -ql -v ERROR --progress_bar none VideoFinalCompleteCNN

Manim Community v0.21.0